# 🚀 Complete Blood Pressure Estimation C Pipeline Generator (`bp_pipeline`)

This notebook executes the prerequisite training and filter tuning notebooks:
1. `models/train_filter.ipynb` (PPG 4th-Order Chebyshev II + ECG 3rd-Order Butterworth + Pan-Tompkins @ 100 Hz)
2. `models/train_red_ir.ipynb` (10-second sliding windows, `ECG_I_Filtered` + Dual PPG, 72 features, LightGBM **SBP and DBP** models exported to C via `m2cgen`)

Then, it generates, compiles, and verifies the complete **`bp_pipeline`** in `deploy/`:
- `deploy/bp_pipeline.h`: Unified C pipeline interface (SBP & DBP)
- `deploy/bp_pipeline.c`: Complete C signal processing, landmark detection, 72-feature extraction, and `m2cgen` inference
- `deploy/predict_bp_example.c`: Standalone runnable demonstration program with real clinical 10-second test window
- `deploy/Makefile`: Production build automation
- **Clinical Subject Validation Test**: Validates the ported C pipeline on a subject from the dataset.


In [ ]:
import os
import sys
import json
import glob
import shutil
import subprocess
import numpy as np
import pandas as pd
import scipy.signal as signal

# Project root resolution
root_dir = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(root_dir, "config", "filter_config.json")):
    parent = os.path.dirname(root_dir)
    if parent == root_dir:
        break
    root_dir = parent

config_dir = os.path.join(root_dir, "config")
plots_dir = os.path.join(root_dir, "plots")
deploy_dir = os.path.join(root_dir, "deploy")
models_dir = os.path.join(root_dir, "models")
data_dir = os.path.join(root_dir, "A dataset of simultaneous collected ECG and PPG signals")

os.makedirs(deploy_dir, exist_ok=True)
print(f"Environment initialized. Project root: {root_dir}")


## Step 1: Run Prerequisite Notebooks (`train_filter.ipynb` & `train_red_ir.ipynb`)


In [ ]:
print("================================================================================")
print(" 1. Running models/train_filter.ipynb (PPG & ECG Filter Tuning & SOS Export)")
print("================================================================================")
filter_nb_path = os.path.join(models_dir, "train_filter.ipynb")
res_filter = subprocess.run(
    f"jupyter nbconvert --execute --inplace '{filter_nb_path}'",
    shell=True, capture_output=True, text=True
)
if res_filter.returncode != 0:
    print("Warning during train_filter execution:", res_filter.stderr[-500:])
else:
    print("Successfully executed models/train_filter.ipynb")

print("\n================================================================================")
print(" 2. Running models/train_red_ir.ipynb (Feature Extraction, Training & C Export for SBP/DBP)")
print("================================================================================")
train_nb_path = os.path.join(models_dir, "train_red_ir.ipynb")
res_train = subprocess.run(
    f"jupyter nbconvert --execute --inplace '{train_nb_path}'",
    shell=True, capture_output=True, text=True
)
if res_train.returncode != 0:
    print("Warning during train_red_ir execution:", res_train.stderr[-500:])
else:
    print("Successfully executed models/train_red_ir.ipynb")

# Verify essential files are present
required_files = [
    "lgbm_sbp.c", "lgbm_sbp.h",
    "lgbm_dbp.c", "lgbm_dbp.h",
    "bp_models.h", "feature_names.json",
    "ppg_bandpass_filter.h", "ppg_filter.c",
    "ecg_filter.h", "ecg_filter.c"
]

for rf in required_files:
    p = os.path.join(deploy_dir, rf)
    if not os.path.exists(p):
        src = os.path.join(config_dir, rf)
        if os.path.exists(src):
            shutil.copy2(src, p)
    print(f"  [OK] {rf:<24} ({os.path.getsize(os.path.join(deploy_dir, rf)):,} bytes)")


## Step 2: Generate C Blood Pressure Pipeline (`bp_pipeline.h` & `bp_pipeline.c`)


In [ ]:
# 1. Write deploy/bp_pipeline.h
h_code = '/**\n * @file bp_pipeline.h\n * @brief End-to-End Blood Pressure Preprocessing, Feature Extraction, and Inference Pipeline (SBP & DBP)\n */\n\n#ifndef BP_PIPELINE_H\n#define BP_PIPELINE_H\n\n#include <stddef.h>\n#include <stdbool.h>\n#include "bp_models.h"\n#include "ppg_bandpass_filter.h"\n#include "ecg_filter.h"\n\n#ifdef __cplusplus\nextern "C" {\n#endif\n\n#define BP_SAMPLING_RATE_HZ 100.0\n#define BP_WINDOW_SAMPLES   1000  // 10.0 seconds at 100 Hz\n\ntypedef struct {\n    double sbp;               /**< Systolic Blood Pressure (mmHg) */\n    double dbp;               /**< Diastolic Blood Pressure (mmHg) */\n    double map_calc;          /**< Analytically Calculated Mean Arterial Pressure (DBP + 1/3*(SBP-DBP)) */\n    double heart_rate_bpm;    /**< Estimated Heart Rate (BPM) */\n    double pat_foot_ms;       /**< Mean Pulse Arrival Time to Foot (ms) */\n    double pat_peak_ms;       /**< Mean Pulse Arrival Time to Peak (ms) */\n    double ptt_inter_peak_ms; /**< Inter-channel Red-IR Transit Time (ms) */\n    double pulse_width_50_ms; /**< PPG 50% Pulse Width (ms) */\n    double stiffness_k_val;   /**< Vascular Stiffness Index (Tsys / Tdia) */\n    double optical_ratio_r;   /**< Red / IR Optical Ratio */\n    int num_detected_beats;   /**< Number of valid detected cardiac cycles */\n    const char *aha_category; /**< AHA / ACC Blood Pressure Classification String */\n} bp_prediction_result_t;\n\nvoid bp_min_max_normalize(double *buffer, size_t length);\nvoid bp_compute_derivatives(const double *input, double *v_out, double *a_out, size_t length);\n\nbool bp_extract_features(\n    const double *raw_ppg_red,\n    const double *raw_ppg_ir,\n    const double *raw_ecg,\n    size_t num_samples,\n    double is_male,\n    double features_out[NUM_INPUT_FEATURES]\n);\n\nbool bp_predict_from_raw(\n    const double *raw_ppg_red,\n    const double *raw_ppg_ir,\n    const double *raw_ecg,\n    size_t num_samples,\n    double is_male,\n    bp_prediction_result_t *result\n);\n\nconst char *bp_get_aha_classification(double sbp, double dbp);\n\n#ifdef __cplusplus\n}\n#endif\n\n#endif // BP_PIPELINE_H\n'
with open(os.path.join(deploy_dir, "bp_pipeline.h"), "w") as f:
    f.write(h_code)
print(f"Generated header: {os.path.join(deploy_dir, 'bp_pipeline.h')}")

# 2. Write deploy/bp_pipeline.c
c_code = '/**\n * @file bp_pipeline.c\n * @brief Implementation of End-to-End Blood Pressure Preprocessing, Feature Extraction, and Inference (SBP & DBP)\n */\n\n#include "bp_pipeline.h"\n#include "ppg_bandpass_filter.h"\n#include "ecg_filter.h"\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <math.h>\n\nvoid bp_min_max_normalize(double *buffer, size_t length) {\n    if (length == 0) return;\n    double min_v = buffer[0];\n    double max_v = buffer[0];\n    for (size_t i = 1; i < length; i++) {\n        if (buffer[i] < min_v) min_v = buffer[i];\n        if (buffer[i] > max_v) max_v = buffer[i];\n    }\n    double range = max_v - min_v;\n    if (range > 1e-9) {\n        for (size_t i = 0; i < length; i++) {\n            buffer[i] = (buffer[i] - min_v) / range;\n        }\n    } else {\n        for (size_t i = 0; i < length; i++) {\n            buffer[i] = 0.0;\n        }\n    }\n}\n\nvoid bp_compute_derivatives(const double *input, double *v_out, double *a_out, size_t length) {\n    if (length < 3) return;\n    v_out[0] = input[1] - input[0];\n    for (size_t i = 1; i < length - 1; i++) {\n        v_out[i] = (input[i + 1] - input[i - 1]) / 2.0;\n    }\n    v_out[length - 1] = input[length - 1] - input[length - 2];\n\n    a_out[0] = v_out[1] - v_out[0];\n    for (size_t i = 1; i < length - 1; i++) {\n        a_out[i] = (v_out[i + 1] - v_out[i - 1]) / 2.0;\n    }\n    a_out[length - 1] = v_out[length - 1] - v_out[length - 2];\n}\n\nstatic int find_peaks_1d(const double *sig, size_t length, int min_dist, double min_prom, int *peaks_out, int max_peaks) {\n    int candidates[256];\n    int cand_count = 0;\n\n    for (size_t i = 1; i < length - 1 && cand_count < 256; i++) {\n        if (sig[i] > sig[i - 1] && sig[i] >= sig[i + 1]) {\n            double left_min = sig[i];\n            for (int k = (int)i - 1; k >= 0; k--) {\n                if (sig[k] > sig[i]) break;\n                if (sig[k] < left_min) left_min = sig[k];\n            }\n\n            double right_min = sig[i];\n            for (size_t k = i + 1; k < length; k++) {\n                if (sig[k] > sig[i]) break;\n                if (sig[k] < right_min) right_min = sig[k];\n            }\n\n            double higher_valley = (left_min > right_min) ? left_min : right_min;\n            double prom = sig[i] - higher_valley;\n\n            if (prom >= min_prom) {\n                candidates[cand_count++] = (int)i;\n            }\n        }\n    }\n\n    if (cand_count == 0) return 0;\n\n    int order[256];\n    for (int i = 0; i < cand_count; i++) order[i] = i;\n    for (int i = 0; i < cand_count - 1; i++) {\n        for (int j = i + 1; j < cand_count; j++) {\n            if (sig[candidates[order[j]]] > sig[candidates[order[i]]]) {\n                int tmp = order[i];\n                order[i] = order[j];\n                order[j] = tmp;\n            }\n        }\n    }\n\n    int kept[256];\n    int kept_count = 0;\n    for (int i = 0; i < cand_count && kept_count < max_peaks; i++) {\n        int c = candidates[order[i]];\n        bool keep = true;\n        for (int k = 0; k < kept_count; k++) {\n            if (abs(c - kept[k]) < min_dist) {\n                keep = false;\n                break;\n            }\n        }\n        if (keep) {\n            kept[kept_count++] = c;\n        }\n    }\n\n    for (int i = 0; i < kept_count - 1; i++) {\n        for (int j = i + 1; j < kept_count; j++) {\n            if (kept[i] > kept[j]) {\n                int tmp = kept[i];\n                kept[i] = kept[j];\n                kept[j] = tmp;\n            }\n        }\n    }\n\n    for (int i = 0; i < kept_count; i++) {\n        peaks_out[i] = kept[i];\n    }\n    return kept_count;\n}\n\ntypedef struct {\n    double pw25, pw50, pw75;\n    double area_a1, area_a2, area_ratio, ipa_ratio;\n    double aix, decay_slope, tsys, tdia;\n    double apg_b_a, apg_agi;\n} morph_features_t;\n\nstatic morph_features_t compute_single_pulse_morphology(const double *pulse, size_t pulse_len) {\n    morph_features_t m;\n    memset(&m, 0, sizeof(m));\n    if (pulse_len < 20) return m;\n\n    size_t p_peak = 0;\n    double max_p = pulse[0], min_p = pulse[0];\n    for (size_t i = 1; i < pulse_len; i++) {\n        if (pulse[i] > max_p) {\n            max_p = pulse[i];\n            p_peak = i;\n        }\n        if (pulse[i] < min_p) {\n            min_p = pulse[i];\n        }\n    }\n    double p_range = max_p - min_p;\n    if (p_range < 1e-5) return m;\n\n    m.tsys = ((double)p_peak / BP_SAMPLING_RATE_HZ) * 1000.0;\n    m.tdia = (((double)(pulse_len - 1 - p_peak)) / BP_SAMPLING_RATE_HZ) * 1000.0;\n\n    double *p_norm = (double *)malloc(pulse_len * sizeof(double));\n    if (!p_norm) return m;\n    for (size_t i = 0; i < pulse_len; i++) {\n        p_norm[i] = (pulse[i] - min_p) / p_range;\n    }\n\n    int c25 = 0, c50 = 0, c75 = 0;\n    double sum_a1 = 0.0, sum_a2 = 0.0;\n    for (size_t i = 0; i < pulse_len; i++) {\n        if (p_norm[i] >= 0.25) c25++;\n        if (p_norm[i] >= 0.50) c50++;\n        if (p_norm[i] >= 0.75) c75++;\n        if (i <= p_peak) sum_a1 += p_norm[i];\n        else sum_a2 += p_norm[i];\n    }\n    m.pw25 = ((double)c25 / BP_SAMPLING_RATE_HZ) * 1000.0;\n    m.pw50 = ((double)c50 / BP_SAMPLING_RATE_HZ) * 1000.0;\n    m.pw75 = ((double)c75 / BP_SAMPLING_RATE_HZ) * 1000.0;\n    m.area_a1 = sum_a1;\n    m.area_a2 = sum_a2;\n    m.area_ratio = sum_a1 / (sum_a2 + 1e-5);\n    m.ipa_ratio = sum_a2 / (sum_a1 + sum_a2 + 1e-5);\n\n    size_t decay_len = pulse_len - 1 - p_peak;\n    if (decay_len > 0) {\n        m.decay_slope = (p_norm[pulse_len - 1] - p_norm[p_peak]) / ((double)decay_len / BP_SAMPLING_RATE_HZ + 1e-5);\n    }\n\n    double *v = (double *)malloc(pulse_len * sizeof(double));\n    double *a = (double *)malloc(pulse_len * sizeof(double));\n    if (v && a) {\n        bp_compute_derivatives(p_norm, v, a, pulse_len);\n\n        int a_peaks[32];\n        int n_a_p = find_peaks_1d(a, pulse_len, 3, 0.001, a_peaks, 32);\n        double a_val = (n_a_p > 0) ? a[a_peaks[0]] : a[0];\n        if (fabs(a_val) < 1e-5) a_val = 1.0;\n\n        double b_val = a[0];\n        for (size_t i = 0; i <= p_peak && i < pulse_len; i++) {\n            if (a[i] < b_val) b_val = a[i];\n        }\n        m.apg_b_a = b_val / (fabs(a_val) + 1e-5);\n        m.apg_agi = b_val / (fabs(a_val) + 1e-5);\n\n        double *neg_v = (double *)malloc(pulse_len * sizeof(double));\n        if (neg_v) {\n            for (size_t i = 0; i < pulse_len; i++) neg_v[i] = -v[i];\n            int v_valleys[32];\n            int n_vv = find_peaks_1d(neg_v, pulse_len, 3, 0.001, v_valleys, 32);\n            int notch_idx = -1;\n            for (int k = 0; k < n_vv; k++) {\n                if ((size_t)v_valleys[k] > p_peak) {\n                    notch_idx = v_valleys[k];\n                    break;\n                }\n            }\n            if (notch_idx >= 0 && (size_t)notch_idx < pulse_len - 1) {\n                size_t dia_peak = (size_t)notch_idx;\n                double max_dia = p_norm[dia_peak];\n                for (size_t i = (size_t)notch_idx; i < pulse_len; i++) {\n                    if (p_norm[i] > max_dia) {\n                        max_dia = p_norm[i];\n                        dia_peak = i;\n                    }\n                }\n                m.aix = (p_norm[dia_peak] - p_norm[p_peak]) / (p_norm[p_peak] + 1e-5);\n            } else {\n                m.aix = 0.0;\n            }\n            free(neg_v);\n        }\n        free(v); free(a);\n    }\n    free(p_norm);\n    return m;\n}\n\nstatic morph_features_t compute_morphology(const double *sig, size_t length) {\n    double *neg_sig = (double *)malloc(length * sizeof(double));\n    if (!neg_sig) return compute_single_pulse_morphology(sig, length);\n    for (size_t i = 0; i < length; i++) neg_sig[i] = -sig[i];\n\n    int feet[64];\n    int n_feet = find_peaks_1d(neg_sig, length, (int)(0.35 * BP_SAMPLING_RATE_HZ), 0.02, feet, 64);\n    free(neg_sig);\n\n    if (n_feet < 2) {\n        return compute_single_pulse_morphology(sig, length);\n    }\n\n    morph_features_t avg;\n    memset(&avg, 0, sizeof(avg));\n    int valid_count = 0;\n\n    for (int i = 0; i < n_feet - 1; i++) {\n        int f_start = feet[i];\n        int f_end = feet[i + 1];\n        int plen = f_end - f_start;\n        if (plen >= 35) {\n            morph_features_t single = compute_single_pulse_morphology(sig + f_start, (size_t)plen);\n            avg.pw25 += single.pw25;\n            avg.pw50 += single.pw50;\n            avg.pw75 += single.pw75;\n            avg.area_a1 += single.area_a1;\n            avg.area_a2 += single.area_a2;\n            avg.area_ratio += single.area_ratio;\n            avg.ipa_ratio += single.ipa_ratio;\n            avg.aix += single.aix;\n            avg.decay_slope += single.decay_slope;\n            avg.tsys += single.tsys;\n            avg.tdia += single.tdia;\n            avg.apg_b_a += single.apg_b_a;\n            avg.apg_agi += single.apg_agi;\n            valid_count++;\n        }\n    }\n\n    if (valid_count > 0) {\n        avg.pw25 /= valid_count;\n        avg.pw50 /= valid_count;\n        avg.pw75 /= valid_count;\n        avg.area_a1 /= valid_count;\n        avg.area_a2 /= valid_count;\n        avg.area_ratio /= valid_count;\n        avg.ipa_ratio /= valid_count;\n        avg.aix /= valid_count;\n        avg.decay_slope /= valid_count;\n        avg.tsys /= valid_count;\n        avg.tdia /= valid_count;\n        avg.apg_b_a /= valid_count;\n        avg.apg_agi /= valid_count;\n        return avg;\n    }\n\n    return compute_single_pulse_morphology(sig, length);\n}\n\nbool bp_extract_features(\n    const double *raw_ppg_red,\n    const double *raw_ppg_ir,\n    const double *raw_ecg,\n    size_t num_samples,\n    double is_male,\n    double features_out[NUM_INPUT_FEATURES]\n) {\n    if (num_samples < BP_WINDOW_SAMPLES) return false;\n\n    double *ir_filt = (double *)malloc(num_samples * sizeof(double));\n    double *red_filt = (double *)malloc(num_samples * sizeof(double));\n    double *ecg_filt = (double *)malloc(num_samples * sizeof(double));\n    double *v_ir = (double *)malloc(num_samples * sizeof(double));\n    double *a_ir = (double *)malloc(num_samples * sizeof(double));\n    double *v_red = (double *)malloc(num_samples * sizeof(double));\n    double *a_red = (double *)malloc(num_samples * sizeof(double));\n\n    double *ecg_diff = (double *)malloc(num_samples * sizeof(double));\n    double *ecg_qrs  = (double *)malloc(num_samples * sizeof(double));\n\n    if (!ir_filt || !red_filt || !ecg_filt || !v_ir || !a_ir || !v_red || !a_red || !ecg_diff || !ecg_qrs) {\n        free(ir_filt); free(red_filt); free(ecg_filt);\n        free(v_ir); free(a_ir); free(v_red); free(a_red);\n        free(ecg_diff); free(ecg_qrs);\n        return false;\n    }\n\n    // 1. Filter Signals via 100 Hz SOS Engines (Zero-Phase FiltFilt)\n    ppg_filtfilt(raw_ppg_ir, ir_filt, (int)num_samples);\n    ppg_filtfilt(raw_ppg_red, red_filt, (int)num_samples);\n    ecg_filtfilt(raw_ecg, ecg_filt, (int)num_samples);\n\n    bp_min_max_normalize(ir_filt, num_samples);\n    bp_min_max_normalize(red_filt, num_samples);\n    bp_min_max_normalize(ecg_filt, num_samples);\n\n    // 2. Pan-Tompkins QRS Energy Integration for ECG R-Peaks\n    ecg_diff[0] = ecg_filt[1] - ecg_filt[0];\n    for (size_t i = 1; i < num_samples - 1; i++) {\n        ecg_diff[i] = (ecg_filt[i + 1] - ecg_filt[i - 1]) / 2.0;\n    }\n    ecg_diff[num_samples - 1] = ecg_filt[num_samples - 1] - ecg_filt[num_samples - 2];\n\n    for (size_t i = 0; i < num_samples; i++) {\n        double d2 = ecg_diff[i] * ecg_diff[i];\n        double sum = 0.0;\n        int count = 0;\n        for (int k = -7; k <= 7; k++) {\n            int idx = (int)i + k;\n            if (idx >= 0 && idx < (int)num_samples) {\n                double diff_k = (idx == 0) ? (ecg_filt[1] - ecg_filt[0]) :\n                                (idx == (int)num_samples - 1) ? (ecg_filt[idx] - ecg_filt[idx-1]) :\n                                (ecg_filt[idx+1] - ecg_filt[idx-1]) / 2.0;\n                sum += diff_k * diff_k;\n                count++;\n            }\n        }\n        ecg_qrs[i] = (count > 0) ? (sum / (double)count) : d2;\n    }\n    bp_min_max_normalize(ecg_qrs, num_samples);\n\n    // 3. Peak Detection\n    int ecg_peaks[128], ir_peaks[128], red_peaks[128];\n    int ecg_cnt = find_peaks_1d(ecg_qrs, num_samples, (int)(0.35 * BP_SAMPLING_RATE_HZ), 0.10, ecg_peaks, 128);\n    int ir_cnt  = find_peaks_1d(ir_filt,  num_samples, (int)(0.35 * BP_SAMPLING_RATE_HZ), 0.05, ir_peaks,  128);\n    int red_cnt = find_peaks_1d(red_filt, num_samples, (int)(0.35 * BP_SAMPLING_RATE_HZ), 0.05, red_peaks, 128);\n\n    if (ecg_cnt < 2 || ir_cnt < 2 || red_cnt < 2) {\n        free(ir_filt); free(red_filt); free(ecg_filt);\n        free(v_ir); free(a_ir); free(v_red); free(a_red);\n        free(ecg_diff); free(ecg_qrs);\n        return false;\n    }\n\n    bp_compute_derivatives(ir_filt, v_ir, a_ir, num_samples);\n    bp_compute_derivatives(red_filt, v_red, a_red, num_samples);\n\n    double pat_p_ir[128], pat_f_ir[128], pat_d_ir[128];\n    double pat_p_red[128], pat_f_red[128], pat_d_red[128];\n    double ptt_inter_p[128], ptt_inter_f[128];\n    int n_pat_ir = 0, n_pat_red = 0, n_ptt = 0;\n\n    for (int e = 0; e < ecg_cnt; e++) {\n        int r_i = ecg_peaks[e];\n        int p_ir_idx = -1, p_red_idx = -1;\n        int f_ir_idx = -1, f_red_idx = -1;\n\n        for (int p = 0; p < ir_cnt; p++) {\n            if (ir_peaks[p] > r_i && ir_peaks[p] < r_i + (int)(0.60 * BP_SAMPLING_RATE_HZ)) {\n                p_ir_idx = ir_peaks[p];\n                break;\n            }\n        }\n        for (int p = 0; p < red_cnt; p++) {\n            if (red_peaks[p] > r_i && red_peaks[p] < r_i + (int)(0.60 * BP_SAMPLING_RATE_HZ)) {\n                p_red_idx = red_peaks[p];\n                break;\n            }\n        }\n\n        if (p_ir_idx >= 0) {\n            pat_p_ir[n_pat_ir] = (double)(p_ir_idx - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n            int s_ir = (p_ir_idx - (int)(0.30 * BP_SAMPLING_RATE_HZ) > 0) ? (p_ir_idx - (int)(0.30 * BP_SAMPLING_RATE_HZ)) : 0;\n            f_ir_idx = s_ir;\n            double min_v = ir_filt[s_ir];\n            for (int k = s_ir + 1; k < p_ir_idx; k++) {\n                if (ir_filt[k] < min_v) {\n                    min_v = ir_filt[k];\n                    f_ir_idx = k;\n                }\n            }\n            pat_f_ir[n_pat_ir] = (double)(f_ir_idx - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n\n            if (f_ir_idx < p_ir_idx) {\n                int d_i = f_ir_idx;\n                double max_dv = v_ir[f_ir_idx];\n                for (int k = f_ir_idx + 1; k < p_ir_idx; k++) {\n                    if (v_ir[k] > max_dv) {\n                        max_dv = v_ir[k];\n                        d_i = k;\n                    }\n                }\n                pat_d_ir[n_pat_ir] = (double)(d_i - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n            } else {\n                pat_d_ir[n_pat_ir] = (pat_f_ir[n_pat_ir] + pat_p_ir[n_pat_ir]) / 2.0;\n            }\n            n_pat_ir++;\n        }\n\n        if (p_red_idx >= 0) {\n            pat_p_red[n_pat_red] = (double)(p_red_idx - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n            int s_red = (p_red_idx - (int)(0.30 * BP_SAMPLING_RATE_HZ) > 0) ? (p_red_idx - (int)(0.30 * BP_SAMPLING_RATE_HZ)) : 0;\n            f_red_idx = s_red;\n            double min_vr = red_filt[s_red];\n            for (int k = s_red + 1; k < p_red_idx; k++) {\n                if (red_filt[k] < min_vr) {\n                    min_vr = red_filt[k];\n                    f_red_idx = k;\n                }\n            }\n            pat_f_red[n_pat_red] = (double)(f_red_idx - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n\n            if (f_red_idx < p_red_idx) {\n                int d_i_r = f_red_idx;\n                double max_dvr = v_red[f_red_idx];\n                for (int k = f_red_idx + 1; k < p_red_idx; k++) {\n                    if (v_red[k] > max_dvr) {\n                        max_dvr = v_red[k];\n                        d_i_r = k;\n                    }\n                }\n                pat_d_red[n_pat_red] = (double)(d_i_r - r_i) / BP_SAMPLING_RATE_HZ * 1000.0;\n            } else {\n                pat_d_red[n_pat_red] = (pat_f_red[n_pat_red] + pat_p_red[n_pat_red]) / 2.0;\n            }\n            n_pat_red++;\n        }\n\n        if (p_ir_idx >= 0 && p_red_idx >= 0) {\n            ptt_inter_p[n_ptt] = (double)(p_red_idx - p_ir_idx) / BP_SAMPLING_RATE_HZ * 1000.0;\n            if (f_ir_idx >= 0 && f_red_idx >= 0) {\n                ptt_inter_f[n_ptt] = (double)(f_red_idx - f_ir_idx) / BP_SAMPLING_RATE_HZ * 1000.0;\n            } else {\n                ptt_inter_f[n_ptt] = ptt_inter_p[n_ptt];\n            }\n            n_ptt++;\n        }\n    }\n\n    if (n_pat_ir == 0) {\n        free(ir_filt); free(red_filt); free(ecg_filt);\n        free(v_ir); free(a_ir); free(v_red); free(a_red);\n        free(ecg_diff); free(ecg_qrs);\n        return false;\n    }\n\n    double sum_p = 0.0, sum_f = 0.0, sum_d = 0.0;\n    for (int i = 0; i < n_pat_ir; i++) {\n        sum_p += pat_p_ir[i];\n        sum_f += pat_f_ir[i];\n        sum_d += pat_d_ir[i];\n    }\n    double pat_p_mean = sum_p / n_pat_ir;\n    double pat_f_mean = sum_f / n_pat_ir;\n    double pat_d_mean = sum_d / n_pat_ir;\n\n    double ptt_inter_peak = 0.0, ptt_inter_foot = 0.0;\n    if (n_ptt > 0) {\n        for (int i = 0; i < n_ptt; i++) {\n            ptt_inter_peak += ptt_inter_p[i];\n            ptt_inter_foot += ptt_inter_f[i];\n        }\n        ptt_inter_peak /= n_ptt;\n        ptt_inter_foot /= n_ptt;\n    }\n\n    double delta_pat_peak_red_ir = 0.0, delta_pat_foot_red_ir = 0.0, delta_pat_deriv_red_ir = 0.0;\n    if (n_pat_red > 0) {\n        double sum_pr = 0.0, sum_fr = 0.0, sum_dr = 0.0;\n        for (int i = 0; i < n_pat_red; i++) {\n            sum_pr += pat_p_red[i];\n            sum_fr += pat_f_red[i];\n            sum_dr += pat_d_red[i];\n        }\n        delta_pat_peak_red_ir = (sum_pr / n_pat_red) - pat_p_mean;\n        delta_pat_foot_red_ir = (sum_fr / n_pat_red) - pat_f_mean;\n        delta_pat_deriv_red_ir = (sum_dr / n_pat_red) - pat_d_mean;\n    }\n\n    double rr_sec = 0.8;\n    if (ecg_cnt >= 2) {\n        double total_rr = 0.0;\n        for (int i = 1; i < ecg_cnt; i++) {\n            total_rr += (double)(ecg_peaks[i] - ecg_peaks[i-1]);\n        }\n        rr_sec = (total_rr / (double)(ecg_cnt - 1)) / BP_SAMPLING_RATE_HZ;\n    }\n    if (rr_sec < 0.3) rr_sec = 0.3;\n    if (rr_sec > 2.0) rr_sec = 2.0;\n\n    double pep_est = 60.0 + 0.12 * (1000.0 * rr_sec) * 0.1;\n    double ptt_p_est = pat_p_mean - pep_est;\n    double ptt_f_est = pat_f_mean - pep_est;\n    double ptt_d_est = pat_d_mean - pep_est;\n\n    double pat_f_bazett = pat_f_mean / sqrt(rr_sec + 1e-5);\n    double pat_d_bazett = pat_d_mean / sqrt(rr_sec + 1e-5);\n    double pat_p_bazett = pat_p_mean / sqrt(rr_sec + 1e-5);\n\n    double pat_f_fridericia = pat_f_mean / (cbrt(rr_sec) + 1e-5);\n    double pat_d_fridericia = pat_d_mean / (cbrt(rr_sec) + 1e-5);\n    double pat_p_fridericia = pat_p_mean / (cbrt(rr_sec) + 1e-5);\n\n    double pat_f_framingham = pat_f_mean + 0.154 * (1.0 - rr_sec) * 1000.0;\n    double pat_d_framingham = pat_d_mean + 0.154 * (1.0 - rr_sec) * 1000.0;\n    double pat_p_framingham = pat_p_mean + 0.154 * (1.0 - rr_sec) * 1000.0;\n\n    double pat_f_inv = 1.0 / (pat_f_mean + 1e-5);\n    double pat_f_sq_inv = 1.0 / (pat_f_mean * pat_f_mean + 1e-5);\n    double pat_d_inv = 1.0 / (pat_d_mean + 1e-5);\n    double pat_d_sq_inv = 1.0 / (pat_d_mean * pat_d_mean + 1e-5);\n    double pat_p_inv = 1.0 / (pat_p_mean + 1e-5);\n    double pat_p_sq_inv = 1.0 / (pat_p_mean * pat_p_mean + 1e-5);\n\n    double ptt_f_inv = 1.0 / (ptt_f_est + 1e-5);\n    double ptt_d_inv = 1.0 / (ptt_d_est + 1e-5);\n    double ptt_p_inv = 1.0 / (ptt_p_est + 1e-5);\n    double ptt_f_sq_inv = 1.0 / (ptt_f_est * ptt_f_est + 1e-5);\n    double ptt_d_sq_inv = 1.0 / (ptt_d_est * ptt_d_est + 1e-5);\n    double ptt_p_sq_inv = 1.0 / (ptt_p_est * ptt_p_est + 1e-5);\n\n    double t_sys_dia = pat_p_mean - pat_f_mean;\n    double t_sys_deriv = pat_d_mean - pat_f_mean;\n    double t_deriv_dia = pat_p_mean - pat_d_mean;\n\n    // AC/DC dynamics on raw signals\n    double min_raw_ir = raw_ppg_ir[0], max_raw_ir = raw_ppg_ir[0], sum_raw_ir = 0.0;\n    double min_raw_red = raw_ppg_red[0], max_raw_red = raw_ppg_red[0], sum_raw_red = 0.0;\n    double sum_sq_vir = 0.0, sum_sq_air = 0.0, sum_sq_vred = 0.0, sum_sq_ared = 0.0;\n\n    for (size_t i = 0; i < num_samples; i++) {\n        if (raw_ppg_ir[i] < min_raw_ir) min_raw_ir = raw_ppg_ir[i];\n        if (raw_ppg_ir[i] > max_raw_ir) max_raw_ir = raw_ppg_ir[i];\n        sum_raw_ir += raw_ppg_ir[i];\n\n        if (raw_ppg_red[i] < min_raw_red) min_raw_red = raw_ppg_red[i];\n        if (raw_ppg_red[i] > max_raw_red) max_raw_red = raw_ppg_red[i];\n        sum_raw_red += raw_ppg_red[i];\n\n        sum_sq_vir += v_ir[i] * v_ir[i];\n        sum_sq_air += a_ir[i] * a_ir[i];\n        sum_sq_vred += v_red[i] * v_red[i];\n        sum_sq_ared += a_red[i] * a_red[i];\n    }\n\n    double ac_ir = max_raw_ir - min_raw_ir;\n    double dc_ir = (sum_raw_ir / (double)num_samples) + 1e-5;\n    double pi_ir = (ac_ir / fabs(dc_ir)) * 100.0;\n\n    double ac_red = max_raw_red - min_raw_red;\n    double dc_red = (sum_raw_red / (double)num_samples) + 1e-5;\n    double pi_red = (ac_red / fabs(dc_red)) * 100.0;\n\n    double r_optical_ratio = (ac_red / dc_red) / (ac_ir / dc_ir + 1e-5);\n    double ac_dc_ratio = (ac_red + ac_ir) / (dc_red + dc_ir + 1e-5);\n\n    double ir_vpg_rms = sqrt(sum_sq_vir / (double)num_samples);\n    double ir_apg_rms = sqrt(sum_sq_air / (double)num_samples);\n    double red_vpg_rms = sqrt(sum_sq_vred / (double)num_samples);\n    double red_apg_rms = sqrt(sum_sq_ared / (double)num_samples);\n    double ir_shr = ir_vpg_rms / (ir_apg_rms + 1e-5);\n    double red_shr = red_vpg_rms / (red_apg_rms + 1e-5);\n\n    morph_features_t ir_m = compute_morphology(ir_filt, num_samples);\n    morph_features_t red_m = compute_morphology(red_filt, num_samples);\n    double k_val = ir_m.tsys / (t_sys_dia + 1e-5);\n\n    features_out[FEAT_PAT_F]                  = pat_f_mean;\n    features_out[FEAT_PAT_D]                  = pat_d_mean;\n    features_out[FEAT_PAT_P]                  = pat_p_mean;\n    features_out[FEAT_PAT_F_BAZETT]           = pat_f_bazett;\n    features_out[FEAT_PAT_D_BAZETT]           = pat_d_bazett;\n    features_out[FEAT_PAT_P_BAZETT]           = pat_p_bazett;\n    features_out[FEAT_PAT_F_FRIDERICIA]       = pat_f_fridericia;\n    features_out[FEAT_PAT_D_FRIDERICIA]       = pat_d_fridericia;\n    features_out[FEAT_PAT_P_FRIDERICIA]       = pat_p_fridericia;\n    features_out[FEAT_PAT_F_FRAMINGHAM]       = pat_f_framingham;\n    features_out[FEAT_PAT_D_FRAMINGHAM]       = pat_d_framingham;\n    features_out[FEAT_PAT_P_FRAMINGHAM]       = pat_p_framingham;\n    features_out[FEAT_PAT_F_INV]              = pat_f_inv;\n    features_out[FEAT_PAT_F_SQ_INV]           = pat_f_sq_inv;\n    features_out[FEAT_PAT_D_INV]              = pat_d_inv;\n    features_out[FEAT_PAT_D_SQ_INV]           = pat_d_sq_inv;\n    features_out[FEAT_PAT_P_INV]              = pat_p_inv;\n    features_out[FEAT_PAT_P_SQ_INV]           = pat_p_sq_inv;\n    features_out[FEAT_PTT_P_EST]              = ptt_p_est;\n    features_out[FEAT_PTT_F_EST]              = ptt_f_est;\n    features_out[FEAT_PTT_D_EST]              = ptt_d_est;\n    features_out[FEAT_PTT_F_INV]              = ptt_f_inv;\n    features_out[FEAT_PTT_D_INV]              = ptt_d_inv;\n    features_out[FEAT_PTT_P_INV]              = ptt_p_inv;\n    features_out[FEAT_PTT_F_SQ_INV]           = ptt_f_sq_inv;\n    features_out[FEAT_PTT_D_SQ_INV]           = ptt_d_sq_inv;\n    features_out[FEAT_PTT_P_SQ_INV]           = ptt_p_sq_inv;\n    features_out[FEAT_PTT_INTER_PEAK]         = ptt_inter_peak;\n    features_out[FEAT_PTT_INTER_FOOT]         = ptt_inter_foot;\n    features_out[FEAT_DELTA_PAT_PEAK_RED_IR]  = delta_pat_peak_red_ir;\n    features_out[FEAT_DELTA_PAT_FOOT_RED_IR]  = delta_pat_foot_red_ir;\n    features_out[FEAT_DELTA_PAT_DERIV_RED_IR] = delta_pat_deriv_red_ir;\n    features_out[FEAT_T_SYS_DIA]              = t_sys_dia;\n    features_out[FEAT_T_SYS_DERIV]            = t_sys_deriv;\n    features_out[FEAT_T_DERIV_DIA]            = t_deriv_dia;\n    features_out[FEAT_PW25]                   = ir_m.pw25;\n    features_out[FEAT_PW50]                   = ir_m.pw50;\n    features_out[FEAT_PW75]                   = ir_m.pw75;\n    features_out[FEAT_K_VAL]                  = k_val;\n    features_out[FEAT_AREA_RATIO]             = ir_m.area_ratio;\n    features_out[FEAT_AIX]                    = ir_m.aix;\n    features_out[FEAT_AIX_RED]                = red_m.aix;\n    features_out[FEAT_PI_IR]                  = pi_ir;\n    features_out[FEAT_PI_RED]                 = pi_red;\n    features_out[FEAT_R_OPTICAL_RATIO]        = r_optical_ratio;\n    features_out[FEAT_AC_DC_RATIO]            = ac_dc_ratio;\n    features_out[FEAT_IR_VPG_RMS]             = ir_vpg_rms;\n    features_out[FEAT_IR_APG_RMS]             = ir_apg_rms;\n    features_out[FEAT_RED_VPG_RMS]            = red_vpg_rms;\n    features_out[FEAT_RED_APG_RMS]            = red_apg_rms;\n    features_out[FEAT_IR_SHR]                 = ir_shr;\n    features_out[FEAT_RED_SHR]                = red_shr;\n    features_out[FEAT_IR_TSYS]                = ir_m.tsys;\n    features_out[FEAT_IR_DECAY_SLOPE]         = ir_m.decay_slope;\n    features_out[FEAT_IR_AREA_A1]             = ir_m.area_a1;\n    features_out[FEAT_IR_AREA_A2]             = ir_m.area_a2;\n    features_out[FEAT_IR_IPA_RATIO]           = ir_m.ipa_ratio;\n    features_out[FEAT_IR_APG_B_A]             = ir_m.apg_b_a;\n    features_out[FEAT_IR_APG_AGI]             = ir_m.apg_agi;\n    features_out[FEAT_RED_TSYS]               = red_m.tsys;\n    features_out[FEAT_RED_DECAY_SLOPE]        = red_m.decay_slope;\n    features_out[FEAT_RED_PW25]               = red_m.pw25;\n    features_out[FEAT_RED_PW50]               = red_m.pw50;\n    features_out[FEAT_RED_PW75]               = red_m.pw75;\n    features_out[FEAT_RED_AREA_A1]            = red_m.area_a1;\n    features_out[FEAT_RED_AREA_A2]            = red_m.area_a2;\n    features_out[FEAT_RED_IPA_RATIO]          = red_m.ipa_ratio;\n    features_out[FEAT_RED_APG_B_A]            = red_m.apg_b_a;\n    features_out[FEAT_RED_APG_AGI]            = red_m.apg_agi;\n    features_out[FEAT_SEX]                    = is_male;\n\n    free(ir_filt); free(red_filt); free(ecg_filt);\n    free(v_ir); free(a_ir); free(v_red); free(a_red);\n    free(ecg_diff); free(ecg_qrs);\n    return true;\n}\n\nconst char *bp_get_aha_classification(double sbp, double dbp) {\n    if (sbp > 180.0 || dbp > 120.0) {\n        return "Hypertensive Crisis (Emergency Care Needed)";\n    } else if (sbp >= 140.0 || dbp >= 90.0) {\n        return "Hypertension Stage 2";\n    } else if ((sbp >= 130.0 && sbp <= 139.0) || (dbp >= 80.0 && dbp <= 89.0)) {\n        return "Hypertension Stage 1";\n    } else if (sbp >= 120.0 && sbp <= 129.0 && dbp < 80.0) {\n        return "Elevated Blood Pressure";\n    } else if (sbp < 120.0 && dbp < 80.0) {\n        return "Normal Blood Pressure";\n    } else {\n        return "Indeterminate / Borderline";\n    }\n}\n\nbool bp_predict_from_raw(\n    const double *raw_ppg_red,\n    const double *raw_ppg_ir,\n    const double *raw_ecg,\n    size_t num_samples,\n    double is_male,\n    bp_prediction_result_t *result\n) {\n    if (!result) return false;\n\n    double features[NUM_INPUT_FEATURES];\n    bool ok = bp_extract_features(raw_ppg_red, raw_ppg_ir, raw_ecg, num_samples, is_male, features);\n    if (!ok) return false;\n\n    result->sbp = predict_sbp(features);\n    result->dbp = predict_dbp(features);\n    result->map_calc = result->dbp + (result->sbp - result->dbp) / 3.0;\n\n    double rr_sec = (features[FEAT_PAT_F] / (features[FEAT_PAT_F_BAZETT] + 1e-5));\n    rr_sec = rr_sec * rr_sec;\n    if (rr_sec > 0.3 && rr_sec < 2.0) {\n        result->heart_rate_bpm = 60.0 / rr_sec;\n    } else {\n        result->heart_rate_bpm = 75.0;\n    }\n\n    result->pat_foot_ms       = features[FEAT_PAT_F];\n    result->pat_peak_ms       = features[FEAT_PAT_P];\n    result->ptt_inter_peak_ms = features[FEAT_PTT_INTER_PEAK];\n    result->pulse_width_50_ms = features[FEAT_PW50];\n    result->stiffness_k_val   = features[FEAT_K_VAL];\n    result->optical_ratio_r   = features[FEAT_R_OPTICAL_RATIO];\n    result->num_detected_beats = (int)(num_samples / (BP_SAMPLING_RATE_HZ * rr_sec + 1e-5));\n    result->aha_category      = bp_get_aha_classification(result->sbp, result->dbp);\n\n    return true;\n}\n'
with open(os.path.join(deploy_dir, "bp_pipeline.c"), "w") as f:
    f.write(c_code)
print(f"Generated implementation: {os.path.join(deploy_dir, 'bp_pipeline.c')}")


## Step 3: Generate Standalone Executable Example (`predict_bp_example.c` & `Makefile`)


In [ ]:
# 1. Write deploy/sample_signals.h
with open(os.path.join(deploy_dir, "sample_signals.h"), "w") as f:
    f.write('#ifndef SAMPLE_SIGNALS_H\n#define SAMPLE_SIGNALS_H\n\n#include <stddef.h>\n\n#define SAMPLE_SUBJECT_ID "001"\n#define SAMPLE_SIGNAL_LEN 1000\n#define SAMPLE_SAMPLING_RATE 100.0f\n#define SAMPLE_IS_MALE 1.0\n\n#define SAMPLE_TRUE_SBP 148.0\n#define SAMPLE_TRUE_DBP 90.0\n#define SAMPLE_TRUE_MAP 109.33\n#define SAMPLE_TRUE_HR  103.0\n\nstatic const double SAMPLE_PPG_RED[1000] = {\n    410985.898484, 411071.131409, 411209.449675, 411335.351103, 411360.371859, 411540.452560, 411487.924028, 411591.880598, 411712.488769, 411787.215709, 411993.724705, 412079.507205, 412189.722945, 412227.303377, 412257.327373, 412281.825015, 412003.403359, 411691.228242, 411261.093395, 410791.461290, 410375.856917, 409965.003990, 409736.567827, 409524.271127, 409393.532984, 409275.585166, 409134.951985, 409096.847514, 409135.825770, 409104.619480, 409102.314551, 409006.021704, 408954.907644, 408796.167078, 408693.126583, 408655.515409, 408471.601061, 408386.687256, 408312.938891, 408298.201580, 408320.721983, 408287.989922, 408415.604577, 408436.971228, 408401.471927, 408620.106198, 408598.438441, 408595.051248, 408654.479206, 408635.073874, 408721.767086, 408635.167648, 408653.799260, 408610.284816, 408644.041235, 408750.795669, 408760.939772, 408823.793062, 408827.825529, 408849.108314, 408888.772514, 408804.351993, 408886.152227, 408986.476748, 408972.279631, 409153.610015, 409221.360492, 409352.595093, 409413.510534, 409452.967841, 409572.049016, 409503.721819, 409510.176107, 409529.731233, 409606.878929, 409755.930856, 409728.665547, 409893.594507, 409921.325220, 409932.442283, 410054.452084, 410068.741138, 410136.809774, 410221.396721, 410194.455707, 410346.541757, 410421.997097, 410506.129230, 410582.231695, 410615.670479, 410764.410313, 410820.656078, 410956.939315, 411056.352596, 411095.169517, 411275.953014, 411300.165246, 411428.608037, 411498.959223, 411562.304925, 411765.739945, 411748.799727, 411901.084323, 412029.324307, 412014.810640, 411967.644930, 411686.775622, 411318.341222, 410848.049908, 410349.692393, 409992.927693, 409530.197011, 409306.649199, 409123.886893, 409009.985638, 409001.091450, 408895.576103, 408852.317274, 408751.673309, 408662.265546, 408664.394480, 408518.513798, 408422.901379, 408357.740484, 408255.544804, 408209.546169, 408083.078965, 408059.604260, 407985.524418, 407940.348651, 407985.184308, 407921.041492, 407960.447423, 408099.529494, 408133.209735, 408207.648493, 408203.825146, 408159.031880, 408146.286518, 408105.689151, 408125.906347, 408096.377000, 408113.716986, 408175.351015, 408135.932335, 408204.921149, 408239.327593, 408370.958229, 408432.319649, 408428.469634, 408599.851583, 408587.213976, 408649.824237, 408713.343276, 408714.175906, 408818.463861, 408850.458920, 408940.332255, 408993.877455, 409070.843445, 409212.677578, 409214.274459, 409286.459801, 409315.517295, 409377.615628, 409464.366107, 409510.191290, 409669.900935, 409732.814310, 409799.326279, 409893.081827, 409920.480329, 410003.948691, 410048.255967, 410098.706442, 410265.560401, 410270.537592, 410395.661311, 410506.666363, 410566.413623, 410732.211190, 410734.287448, 410865.262540, 410890.142809, 410959.766813, 411017.034442, 411055.000060, 411219.254003, 411290.401306, 411361.149444, 411535.840940, 411577.646285, 411731.609181, 411881.174999, 411962.475028, 412090.163887, 412109.895633, 412015.816058, 411541.086033, 411088.238993, 410620.916905, 410101.249608, 409752.786379, 409432.689577, 409107.356023, 409053.629881, 409005.743463, 408936.876422, 408913.074343, 408884.276747, 408859.576886, 408705.587042, 408682.727194, 408535.859927, 408381.428284, 408406.680749, 408299.108125, 408280.731348, 408265.389041, 408250.756825, 408285.573266, 408278.018047, 408372.295551, 408382.238681, 408432.140617, 408565.724094, 408649.290221, 408714.142437, 408690.588244, 408732.657226, 408783.974911, 408658.286096, 408629.476929, 408575.811536, 408533.310268, 408560.168829, 408473.114653, 408560.271415, 408675.175685, 408703.683012, 408856.725958, 408872.826826, 408872.396846, 408942.030331, 408934.072560, 409061.373002, 409148.193404, 409235.224774, 409252.261110, 409298.786934, 409512.687221, 409480.872048, 409538.924895, 409682.691847, 409757.579753, 409862.020019, 409890.709034, 410073.991505, 410061.226553, 410115.555421, 410222.850149, 410240.274932, 410326.730757, 410415.300845, 410486.950631, 410693.876711, 410770.417497, 410855.041369, 410840.132506, 410934.314395, 411119.903361, 411109.135329, 411226.713180, 411321.348374, 411376.355304, 411535.840221, 411584.773206, 411736.967812, 411842.443717, 411916.484706, 412112.234649, 412185.871255, 412255.199197, 412395.087465, 412457.216693, 412619.410639, 412603.874496, 412742.658063, 412647.770559, 412725.043513, 412731.696050, 412371.116580, 412000.051375, 411552.691043, 411051.704561, 410701.825410, 410279.039738, 410064.203025, 409905.150230, 409842.695562, 409841.284112, 409728.694814, 409766.319587, 409738.130856, 409663.664116, 409658.266593, 409563.410301, 409397.504576, 409287.447779, 409284.925339, 409258.509092, 409132.534537, 409164.957838, 409207.335937, 409234.966592, 409404.256281, 409422.560185, 409508.111666, 409572.478147, 409555.229742, 409630.291058, 409611.355557, 409655.797672, 409698.182351, 409657.170859, 409687.668909, 409642.740597, 409609.124322, 409615.129630, 409700.281455, 409768.009967, 409680.656508, 409823.944746, 409876.373406, 409876.751684, 410001.191169, 409991.122249, 410082.272754, 410112.613401, 410223.258017, 410318.603801, 410311.340174, 410407.840258, 410475.010420, 410496.464061, 410614.074843, 410656.204474, 410760.898843, 410755.519159, 410861.520081, 411014.430932, 410983.289820, 411114.938342, 411222.571666, 411261.443247, 411403.412680, 411402.119811, 411518.099371, 411550.966974, 411599.581118, 411683.452640, 411677.174886, 411759.569977, 411874.467231, 411990.466218, 412201.345294, 412150.686634, 412277.834136, 412326.771010, 412354.803991, 412553.962246, 412585.557066, 412635.032004, 412808.138013, 412861.815738, 412896.275169, 412633.701546, 412358.940526, 411992.828939, 411542.965928, 411190.199180, 410753.579582, 410551.488999, 410340.943227, 410211.383266, 410184.374669, 410076.472015, 410094.649344, 410050.471248, 409991.029586, 410030.107021, 409832.017322, 409772.913638, 409775.120860, 409619.218271, 409592.998108, 409414.332767, 409384.984682, 409351.899460, 409365.287710, 409468.929631, 409460.708936, 409523.765333, 409543.581040, 409552.700983, 409726.711327, 409704.179481, 409730.447247, 409751.883395, 409719.245328, 409692.400717, 409529.103035, 409616.404417, 409667.184060, 409667.859495, 409781.561366, 409736.467576, 409831.390345, 409889.813077, 409924.777986, 410037.862784, 409999.413646, 410076.463761, 410127.724164, 410145.985209, 410272.587950, 410264.889251, 410324.632859, 410399.658686, 410435.219637, 410491.357232, 410428.603120, 410580.321188, 410684.503695, 410768.429229, 410901.662446, 410952.628251, 411008.941167, 411118.622738, 411144.694945, 411264.429804, 411270.890651, 411391.584513, 411449.884277, 411496.770916, 411646.453717, 411616.778339, 411683.429901, 411733.740628, 411796.855775, 411844.511892, 411826.042435, 412016.573778, 412051.062869, 412077.239717, 412244.915256, 412302.079298, 412312.523402, 412385.417182, 412457.584024, 412610.141480, 412626.951954, 412667.440056, 412699.824493, 412657.965528, 412654.585955, 412323.969612, 411932.403245, 411463.937322, 410988.206638, 410642.240761, 410151.354686, 409896.650347, 409714.062953, 409481.652503, 409503.023977, 409385.551528, 409271.085784, 409242.345091, 409148.288050, 409109.636473, 408968.544476, 408850.754140, 408763.085745, 408625.618325, 408535.561852, 408308.779147, 408184.638040, 408084.800044, 408012.016273, 407988.595380, 407981.153820, 408050.536203, 408092.650935, 408033.629617, 408175.840021, 408116.233965, 408102.559752, 408039.842684, 407993.473476, 407961.088196, 407868.936140, 407840.560303, 407792.923812, 407748.206610, 407784.648416, 407741.628507, 407832.727463, 407874.277014, 407834.395815, 407865.752024, 407837.635779, 407927.444636, 407862.959605, 407916.292727, 408062.224476, 408034.902449, 408078.460599, 408115.255368, 408212.791620, 408343.281380, 408285.287160, 408324.404296, 408307.941521, 408369.659304, 408454.600022, 408457.855252, 408525.906741, 408528.843954, 408596.506065, 408636.201758, 408624.938381, 408734.681275, 408715.528815, 408873.773272, 409058.793268, 409069.480504, 409169.402706, 409182.285568, 409182.807762, 409249.325036, 409235.022568, 409302.263490, 409406.757149, 409482.156360, 409613.939655, 409666.542109, 409779.439666, 409855.606622, 409865.264793, 409921.388770, 409865.625814, 409934.888441, 409938.620291, 409977.671195, 410076.576585, 409961.156062, 409826.663425, 409544.330769, 409024.254556, 408514.794145, 407994.918502, 407550.151267, 407201.526354, 406909.362949, 406703.270345, 406491.618626, 406444.722782, 406358.880892, 406237.311984, 406241.286121, 406056.026999, 405982.920371, 405812.943485, 405674.238215, 405638.001964, 405359.416754, 405255.842455, 405179.845622, 405034.612090, 405015.266639, 405001.961038, 405018.736983, 405065.269411, 405068.177645, 405162.831376, 405146.742908, 405157.981224, 405109.373340, 405082.728507, 405115.861069, 404988.351428, 404977.621739, 404956.811135, 404883.437687, 404967.449699, 404799.805196, 404872.933101, 404914.627512, 404931.117187, 405021.135700, 404971.851273, 405051.724369, 405074.527563, 405133.283198, 405207.253942, 405196.407242, 405301.917112, 405360.211376, 405409.489510, 405572.446253, 405554.970354, 405622.432071, 405698.113933, 405711.470800, 405848.557855, 405775.163463, 405870.346475, 405997.650717, 406013.551555, 406162.654713, 406132.972827, 406216.581626, 406262.938695, 406325.289028, 406406.214304, 406448.305404, 406484.653125, 406578.720766, 406618.109323, 406750.024937, 406829.356277, 406940.141967, 406999.870706, 407019.222456, 407147.649620, 407112.308245, 407204.132320, 407359.725984, 407418.026096, 407616.060344, 407661.151660, 407758.437540, 407794.720386, 407897.099825, 408034.153022, 408082.907676, 408177.505243, 408245.172586, 408302.184839, 408312.323485, 408088.533086, 407768.043439, 407318.842523, 406819.681157, 406430.037761, 405861.320950, 405500.426443, 405289.229241, 405053.901312, 404959.065040, 404855.503831, 404804.734705, 404696.275392, 404568.114823, 404530.944683, 404331.655616, 404209.196467, 404071.089053, 403927.068304, 403868.974447, 403679.364094, 403588.967001, 403508.848735, 403385.442711, 403410.112651, 403402.239467, 403367.065073, 403380.612621, 403436.279174, 403553.060101, 403505.941617, 403519.116189, 403563.865890, 403534.361359, 403613.768691, 403575.862681, 403559.972891, 403542.875798, 403536.774263, 403622.932151, 403605.981204, 403666.760055, 403717.445494, 403715.953574, 403814.325122, 403832.531722, 403917.325000, 403987.216484, 404092.796717, 404266.571335, 404315.744101, 404386.966611, 404462.946764, 404501.849567, 404593.601475, 404607.818959, 404639.531585, 404673.949669, 404747.377495, 404835.790688, 404889.976086, 404982.149093, 405024.781260, 405132.443003, 405271.128928, 405307.755397, 405439.476224, 405509.260416, 405571.880693, 405705.633253, 405719.254669, 405831.700562, 405969.533801, 406076.536875, 406267.140199, 406302.115290, 406421.756300, 406503.098349, 406591.039315, 406770.208347, 406825.680974, 407051.031069, 407142.965855, 407261.191921, 407413.354395, 407383.192543, 407545.577412, 407638.151246, 407774.359714, 407953.904777, 408040.722369, 408171.444141, 408222.782726, 408066.458948, 407778.069132, 407227.416850, 406752.105157, 406184.273594, 405737.342496, 405417.396362, 405131.833393, 404944.954509, 404852.285476, 404788.406798, 404713.142703, 404561.792136, 404556.122931, 404503.707547, 404418.229758, 404362.111238, 404210.504891, 404134.327617, 404019.116244, 403877.790517, 403833.745891, 403683.132586, 403646.482114, 403616.189604, 403597.758536, 403722.126160, 403704.859248, 403805.907075, 403882.885091, 403854.553976, 403987.358377, 403920.512976, 403954.739331, 404010.637571, 403932.510734, 404040.828005, 403900.556317, 403977.195643, 404100.734858, 404138.217190, 404239.492866, 404186.102693, 404297.069681, 404381.521977, 404403.325635, 404548.045737, 404590.425187, 404709.514487, 404787.329820, 404866.516065, 404987.672255, 405031.588752, 405166.816744, 405243.776550, 405296.677987, 405431.473538, 405458.955468, 405574.421577, 405592.686059, 405658.420668, 405837.810613, 405881.099450, 405962.491907, 406044.616415, 406141.449572, 406308.553803, 406344.420845, 406477.901415, 406602.509400, 406683.359921, 406849.238205, 406892.329898, 406991.945293, 407092.048311, 407130.548556, 407245.436165, 407293.390185, 407482.722626, 407510.060681, 407634.762788, 407860.707388, 407873.515699, 407996.903782, 408077.695739, 408146.959805, 408325.240852, 408405.875479, 408527.141892, 408634.128453, 408441.252707, 408096.845820, 407569.276914, 407064.242090, 406630.635164, 406175.401014, 405992.136545, 405694.340369, 405498.010355, 405390.915899, 405345.911670, 405352.480947, 405216.762162, 405203.751075, 405144.833205, 405028.673444, 405064.879256, 404867.599039, 404761.978637, 404634.468298, 404538.437414, 404550.136188, 404492.771129, 404544.130646, 404610.113024, 404646.908181, 404769.522813, 404748.129722, 404785.172316, 404811.048999, 404842.060214, 404940.233293, 404915.031882, 404878.706501, 404908.519096, 404905.663621, 404984.058249, 404953.396300, 405033.735672, 405077.686662, 405102.576482, 405203.400640, 405268.145068, 405388.681804, 405416.318470, 405432.122724, 405589.943667, 405626.258413, 405714.151396, 405814.176208, 405933.759590, 406069.371820, 406104.929397, 406194.674673, 406315.845366, 406374.055186, 406513.550188, 406516.339502, 406620.727257, 406699.727906, 406801.951447, 407009.570826, 407005.848279, 407152.538832, 407244.429023, 407301.051863, 407483.967740, 407472.159580, 407600.458710, 407681.674467, 407732.234503, 407885.099325, 407953.938042, 408083.812437, 408238.281336, 408348.675784, 408445.789936, 408494.561625, 408621.301897, 408724.759203, 408777.464607, 408988.923547, 408928.077778, 408885.618136, 408585.241078, 408161.533805, 407772.307314, 407220.181293, 406807.821984, 406510.306753, 406076.029581, 405960.558017, 405760.819105, 405694.319040, 405565.366945, 405531.193687, 405516.488541, 405380.185222, 405364.425920, 405274.166787, 405174.811722, 405148.251871, 404925.943646, 404683.069630, 404640.787732, 404593.987904, 404693.873030, 404665.419581, 404720.833451, 404697.550684, 404724.761467, 404941.551012, 404883.845931, 404939.762666, 405049.701377, 405046.370471, 405143.421842, 405065.008961, 405099.608897, 405119.287380, 405113.672271, 405139.127457, 405120.667596, 405200.730110, 405271.148548, 405349.699979, 405477.517218, 405487.262900, 405610.081798, 405702.084188, 405740.646700, 405890.853729, 405912.580189, 406022.042306, 406141.625558, 406332.146651, 406478.682894, 406510.467331, 406598.352992, 406731.158035, 406790.529078, 406988.004990, 407003.156854, 407072.206626, 407151.540816, 407227.565151, 407384.232123, 407440.719155, 407602.442085, 407766.042611, 407873.466235, 408073.010084, 408102.350298, 408174.999356, 408196.570995, 408319.616228, 408499.295445, 408555.137417, 408647.464355, 408789.756657, 408871.779604, 409059.273842, 409084.403869, 409188.674866, 409342.471323, 409476.243076\n};\n\nstatic const double SAMPLE_PPG_IR[1000] = {\n    449244.102272, 449293.585338, 449485.287950, 449615.196221, 449762.583441, 449953.241374, 450015.295975, 450153.353120, 450286.868618, 450439.592411, 450713.847687, 450769.070528, 451003.513725, 451225.077966, 451233.465722, 451220.108577, 450801.591768, 450309.642198, 449677.350006, 448934.537041, 448384.768746, 447762.927779, 447335.546322, 446987.901676, 446689.752411, 446602.499517, 446454.548715, 446383.423129, 446233.940429, 446207.315024, 446196.853606, 445954.519504, 445821.862107, 445627.001938, 445472.342142, 445340.078791, 445054.426669, 444897.289372, 444815.917398, 444780.229512, 444808.632935, 444787.136368, 444874.231810, 444990.822951, 445008.468424, 445183.423452, 445139.750004, 445184.993615, 445146.056266, 445137.601253, 445197.605421, 445103.860209, 445145.061526, 445141.943116, 445200.056114, 445324.018354, 445335.288910, 445421.457567, 445463.849978, 445527.487188, 445607.617011, 445599.668656, 445707.869779, 445844.788672, 445885.244853, 446102.532064, 446168.526784, 446331.406540, 446510.541002, 446542.060224, 446667.159914, 446686.746931, 446730.881776, 446843.112517, 446955.492030, 447126.672805, 447254.573930, 447394.853746, 447486.057468, 447535.621677, 447708.224367, 447791.881823, 447953.998574, 448086.998779, 448160.490192, 448369.871554, 448437.601405, 448558.646647, 448693.838949, 448833.987855, 449031.985710, 449088.223524, 449292.846159, 449456.626679, 449594.539097, 449765.134080, 449853.188457, 450016.355222, 450175.043835, 450271.354655, 450469.777588, 450652.284829, 450801.771874, 450972.261489, 451074.659037, 451147.286976, 450856.296770, 450423.435630, 449779.149190, 449040.308674, 448400.592657, 447717.721723, 447153.911917, 446758.814834, 446431.846524, 446205.452946, 445937.942526, 445824.276057, 445666.604760, 445508.145569, 445447.206644, 445206.017986, 445024.418899, 444857.642714, 444645.049245, 444523.499162, 444309.689609, 444173.533786, 444072.936373, 443982.470458, 444041.143321, 443923.482245, 444057.790910, 444170.016131, 444232.630600, 444366.170048, 444280.527578, 444302.262049, 444243.017347, 444189.605129, 444243.881604, 444161.440043, 444194.041964, 444271.396742, 444259.094882, 444372.303268, 444391.126005, 444542.026336, 444604.743617, 444631.308571, 444805.767692, 444836.615756, 444922.815032, 445034.175687, 445137.213295, 445263.035148, 445327.053342, 445461.214360, 445582.951252, 445697.686966, 445910.025764, 445921.747175, 446082.900928, 446168.369143, 446281.673626, 446431.219357, 446491.598680, 446694.215305, 446848.526080, 446967.085019, 447124.816210, 447232.211633, 447302.473839, 447411.313358, 447558.078831, 447753.139745, 447832.107477, 448002.894528, 448196.977334, 448326.079220, 448527.914721, 448602.792440, 448741.768706, 448851.707131, 449003.008221, 449171.419871, 449229.233132, 449415.845525, 449562.490503, 449706.844193, 449974.326766, 450067.118468, 450291.912883, 450499.696162, 450656.896492, 450898.363853, 450818.984067, 450706.163710, 450362.527914, 449732.353678, 449101.292053, 448287.529831, 447600.397344, 447024.418772, 446482.673879, 446219.778016, 445924.361671, 445845.995066, 445717.736741, 445601.390197, 445568.397487, 445371.341023, 445265.077148, 445068.832365, 444843.593699, 444689.521488, 444502.352507, 444373.473649, 444217.978608, 444148.347883, 444191.000093, 444156.901534, 444229.618984, 444312.924840, 444396.158973, 444567.929997, 444609.238769, 444685.822268, 444737.031254, 444760.719639, 444794.119116, 444668.978827, 444670.207175, 444603.118888, 444583.568068, 444700.098210, 444620.126408, 444756.481898, 444883.612457, 444937.880157, 445155.930993, 445223.178662, 445283.583858, 445392.092905, 445432.928155, 445659.899169, 445720.648926, 445927.396786, 446056.133342, 446139.924521, 446349.901686, 446368.657946, 446519.332187, 446678.513708, 446753.493843, 447030.260167, 447099.724748, 447268.625146, 447395.715287, 447497.863893, 447707.123866, 447778.838889, 447906.783138, 448050.871195, 448197.097829, 448505.251135, 448588.865471, 448752.797178, 448877.242494, 448977.323152, 449240.696150, 449285.199258, 449465.972772, 449629.638771, 449743.866394, 450001.543190, 450117.433478, 450330.145873, 450486.715218, 450625.747694, 450901.263692, 451040.375469, 451187.042253, 451387.281528, 451549.248199, 451788.863477, 451863.014688, 452024.457505, 452109.972031, 452073.381509, 451978.379873, 451447.211617, 450865.850467, 450158.016346, 449421.085822, 448843.950896, 448249.251919, 447879.615815, 447631.541136, 447469.952004, 447412.514425, 447292.489539, 447303.611283, 447252.799072, 447152.058108, 447120.548731, 446890.931078, 446777.745376, 446638.577162, 446440.849833, 446390.183719, 446194.091493, 446135.236438, 446129.434546, 446179.911800, 446357.392684, 446431.475959, 446570.411776, 446661.246022, 446739.854104, 446946.021570, 446951.371848, 446966.796146, 447010.824961, 446982.777219, 447041.577269, 446958.375885, 447031.996662, 447069.674741, 447155.863053, 447297.547225, 447318.395913, 447384.797794, 447508.383798, 447579.550522, 447732.854255, 447800.773742, 447917.494737, 448036.283465, 448152.897400, 448301.000982, 448401.208582, 448543.234280, 448674.705007, 448745.423969, 448917.880968, 448983.028597, 449139.755785, 449273.471755, 449361.893728, 449600.736461, 449629.964705, 449825.904107, 449939.694946, 450037.781668, 450307.099153, 450304.799000, 450464.284769, 450564.357407, 450642.571239, 450803.661822, 450888.621631, 451051.636167, 451258.798405, 451399.871284, 451636.182614, 451725.373710, 451847.007701, 451965.354325, 452111.191137, 452323.037452, 452447.371637, 452599.677296, 452764.158517, 452840.178621, 452792.878401, 452416.483924, 451937.432105, 451329.090872, 450641.887026, 450047.786373, 449457.910334, 449057.376114, 448771.233253, 448558.051686, 448482.674559, 448340.242750, 448287.384691, 448219.947302, 448102.041723, 448112.247419, 447939.413524, 447796.732768, 447658.730591, 447538.547743, 447374.659530, 447201.358567, 447173.374813, 447114.719329, 447144.424303, 447268.418915, 447300.871584, 447393.361398, 447455.519624, 447524.654931, 447684.723766, 447663.996047, 447706.902801, 447739.213846, 447698.362283, 447742.702817, 447661.236376, 447639.374668, 447648.634940, 447674.654717, 447863.307066, 447834.058024, 447949.690030, 448092.048508, 448155.790902, 448321.034849, 448324.077316, 448482.450186, 448655.258095, 448733.401899, 448935.061680, 448939.795529, 449074.066722, 449187.265815, 449261.599748, 449434.316359, 449462.579371, 449593.130445, 449726.814213, 449875.012055, 450039.095768, 450120.006593, 450287.694563, 450401.121736, 450448.847706, 450605.925017, 450725.273644, 450850.806772, 451022.706196, 451129.312903, 451291.936416, 451319.736405, 451438.546643, 451577.108823, 451624.799342, 451843.420550, 451897.637004, 451988.000882, 452140.697769, 452229.789386, 452435.437160, 452518.025862, 452625.319861, 452737.177772, 452872.394584, 453074.931943, 453124.160644, 453283.038404, 453361.717529, 453315.925249, 453177.967644, 452648.329290, 452008.631877, 451238.282642, 450509.956680, 449892.612211, 449240.759073, 448814.516221, 448470.947096, 448188.876406, 448040.144450, 447855.314525, 447692.800514, 447571.392737, 447404.180283, 447327.291507, 447107.913166, 446877.280994, 446697.460851, 446456.773426, 446304.048323, 445969.208319, 445781.524198, 445601.126337, 445521.539148, 445546.363859, 445536.493539, 445635.946121, 445678.531773, 445699.504374, 445766.203320, 445699.956103, 445676.098098, 445601.850017, 445511.975718, 445529.682578, 445398.607338, 445378.368092, 445330.985695, 445254.324020, 445401.163629, 445346.655931, 445377.091430, 445462.071567, 445494.371643, 445557.145567, 445521.852683, 445603.092546, 445640.111949, 445690.819928, 445823.980618, 445875.974392, 446011.724408, 446045.197884, 446100.216106, 446323.066647, 446321.947017, 446412.255990, 446466.410367, 446478.592323, 446718.464092, 446771.787295, 446920.139646, 446939.940570, 447019.445815, 447160.300814, 447159.793673, 447327.379800, 447486.000374, 447619.713899, 447862.942638, 447856.379108, 447997.775251, 448060.726322, 448094.870982, 448234.451666, 448261.049539, 448419.015594, 448547.160990, 448665.960348, 448874.495651, 448964.784926, 449151.973903, 449276.217915, 449302.828103, 449495.518086, 449478.553827, 449588.119783, 449692.872204, 449719.632832, 449855.796968, 449706.359823, 449443.324031, 448861.261189, 447982.918653, 447246.885037, 446445.111588, 445725.787679, 445192.320835, 444730.778785, 444379.417876, 444068.510945, 443912.591314, 443758.306286, 443584.688251, 443515.762420, 443225.738953, 443006.807120, 442811.928038, 442539.737694, 442359.661763, 442002.391099, 441768.091060, 441534.963052, 441362.909003, 441346.791941, 441245.590610, 441264.085456, 441305.333523, 441319.533927, 441445.431945, 441439.949672, 441375.730989, 441392.917386, 441342.996108, 441326.976989, 441206.967930, 441167.126890, 441139.737237, 441058.924570, 441144.084727, 441072.541161, 441095.398833, 441103.556607, 441149.666558, 441273.221616, 441238.227052, 441336.279403, 441390.037738, 441458.025063, 441603.405117, 441626.783613, 441769.120062, 441893.923577, 441973.500173, 442144.902919, 442210.390829, 442306.585269, 442401.251570, 442508.155120, 442635.461031, 442680.470300, 442795.980872, 442917.651266, 443056.638257, 443191.528221, 443248.794184, 443383.436900, 443523.447143, 443537.393431, 443689.684897, 443740.014951, 443890.019460, 444014.781155, 444098.190093, 444358.150050, 444447.149967, 444630.666732, 444742.961555, 444820.629797, 445006.452348, 445072.431098, 445225.118955, 445378.076592, 445538.580224, 445737.183144, 445845.490254, 446033.763235, 446129.867472, 446254.342548, 446492.633780, 446550.937818, 446749.905574, 446861.592020, 446940.567259, 446970.536919, 446611.871489, 446065.515785, 445322.093114, 444541.542768, 443811.675541, 443027.744422, 442482.227565, 442024.108099, 441660.193847, 441459.656884, 441196.579187, 441026.957198, 440872.906249, 440672.241325, 440512.954621, 440250.954812, 440002.884144, 439751.874091, 439488.136525, 439319.635073, 439020.250097, 438822.440345, 438627.869294, 438478.773486, 438427.603330, 438386.901090, 438411.706264, 438407.537939, 438477.820897, 438577.774106, 438567.068821, 438578.812436, 438601.364548, 438531.953864, 438599.497054, 438535.277489, 438522.946212, 438546.217977, 438528.015060, 438612.361177, 438618.135893, 438657.225840, 438719.730829, 438759.057172, 438913.251240, 438940.243331, 439021.533524, 439143.066195, 439303.037825, 439521.776847, 439554.815800, 439687.775940, 439800.369278, 439899.402080, 440035.974380, 440069.631541, 440190.645358, 440281.788387, 440388.425640, 440583.671242, 440654.045253, 440814.162978, 440936.958430, 441057.561714, 441287.851459, 441351.229239, 441526.192516, 441662.353399, 441773.781758, 441951.257163, 442002.262140, 442230.166859, 442415.753467, 442597.606155, 442805.262625, 442874.819412, 443103.555226, 443249.927161, 443413.815446, 443639.520926, 443751.634449, 443994.565210, 444149.261465, 444255.448131, 444487.652049, 444610.555815, 444760.781104, 444914.932915, 445116.654254, 445368.358527, 445491.633780, 445700.919469, 445880.589820, 445848.111051, 445610.591012, 444943.407147, 444198.633469, 443365.658879, 442560.617660, 441967.737604, 441304.590849, 440616.808449, 440372.884089, 440156.869045, 440065.104518, 439851.748531, 439717.095363, 439546.743881, 439356.457297, 439200.842018, 438948.004485, 438763.227380, 438561.630827, 438276.534183, 438139.116819, 437933.000810, 437772.056839, 437718.729530, 437687.372505, 437769.401503, 437806.806444, 437932.389779, 438043.086562, 438018.779627, 438145.732531, 438071.069016, 438107.361958, 438108.781687, 438091.843579, 438132.358153, 438059.440682, 438161.893170, 438207.297146, 438234.013709, 438401.487491, 438377.570320, 438527.676235, 438639.510540, 438659.996224, 438870.350960, 438944.169317, 439097.990774, 439276.321698, 439396.361172, 439546.515534, 439639.718612, 439771.302703, 439885.687572, 439991.159099, 440198.290350, 440270.200123, 440359.876628, 440500.006588, 440633.411845, 440837.470590, 440924.376710, 441097.213421, 441214.547759, 441356.784514, 441584.223136, 441709.065169, 441889.052428, 442007.370353, 442121.727943, 442437.438726, 442482.558003, 442622.325178, 442784.788323, 442910.584905, 443093.473394, 443198.343756, 443406.277427, 443557.792237, 443741.404311, 444000.453430, 444100.671991, 444289.464876, 444463.096949, 444607.514078, 444836.087810, 444964.530270, 445163.790308, 445222.534437, 445134.309109, 444808.032080, 444115.187450, 443397.045947, 442602.980263, 441841.343104, 441373.403602, 440795.777352, 440415.968418, 440157.953713, 439888.653945, 439805.103146, 439568.772339, 439479.760442, 439364.831646, 439204.562868, 439112.045481, 438891.737462, 438695.256921, 438493.210167, 438158.631792, 438009.341066, 437996.425987, 438018.200028, 438088.306296, 438115.252588, 438238.694067, 438235.407571, 438305.604134, 438351.175065, 438362.164753, 438452.303814, 438387.445537, 438384.115923, 438368.958956, 438351.496266, 438424.266640, 438420.054825, 438489.444866, 438566.186330, 438626.865487, 438770.097204, 438837.834558, 438973.121821, 439083.338642, 439119.895661, 439326.892168, 439390.433526, 439533.658699, 439715.778475, 439844.077981, 440066.538535, 440124.217202, 440279.843588, 440423.479295, 440551.073818, 440709.123216, 440779.830815, 440887.987263, 441024.106363, 441225.553601, 441465.845929, 441557.173470, 441711.760903, 441876.795845, 441945.107120, 442168.108478, 442267.183623, 442407.319847, 442548.666334, 442685.069117, 442908.475251, 442994.817751, 443193.287002, 443397.527770, 443519.154837, 443739.736082, 443875.337916, 444059.920186, 444209.388719, 444357.228995, 444543.651761, 444554.895450, 444381.887932, 443918.941966, 443259.558027, 442642.538002, 441804.203426, 441134.643478, 440538.804737, 439991.433887, 439646.607122, 439275.848500, 439074.195601, 438911.667596, 438664.417377, 438616.135125, 438399.424812, 438243.846794, 438109.475103, 437836.092157, 437693.849610, 437378.295411, 437148.613227, 436960.480208, 436824.487629, 436810.145885, 436732.315584, 436748.929824, 436819.182997, 436852.485772, 436958.308199, 436982.631052, 437061.279699, 437078.330410, 437121.987415, 437198.345037, 437186.593327, 437193.228697, 437158.214476, 437146.824043, 437196.171119, 437161.715949, 437299.179942, 437410.325721, 437479.375589, 437637.584458, 437681.998467, 437839.927407, 437980.460373, 438116.898022, 438276.780469, 438333.461183, 438547.996113, 438715.861093, 438899.545040, 439111.508509, 439190.755308, 439357.101534, 439507.086201, 439651.292191, 439886.630050, 439958.466172, 440048.786421, 440160.796717, 440322.420597, 440562.247730, 440627.663783, 440890.901601, 441092.557745, 441245.278022, 441502.750284, 441555.829400, 441740.232639, 441823.062346, 441989.723046, 442263.377756, 442370.632072, 442543.523900, 442714.807783, 442912.955763, 443144.131194, 443230.864806, 443423.308369, 443624.156765, 443840.382634\n};\n\nstatic const double SAMPLE_ECG_LEAD_I[1000] = {\n    364918.515248, 367874.391675, 365973.361828, 359008.777705, 353390.731415, 353267.834266, 356435.699847, 359510.118555, 360497.373487, 360455.317165, 361034.524189, 361681.357082, 361771.357424, 362488.576044, 362581.776580, 362624.857839, 363350.082685, 363632.072601, 364345.076635, 364987.040372, 365720.906850, 366706.902499, 367372.079359, 368111.175279, 369086.435762, 369380.875374, 368781.861773, 368224.186481, 367000.524790, 365307.392486, 363941.805162, 362646.239417, 362136.900284, 362216.236850, 361859.464157, 361693.542351, 361538.250494, 361692.448002, 361136.279964, 360964.319204, 361428.838904, 361877.988716, 361727.927972, 361046.308031, 360738.745670, 361492.629023, 362181.815233, 362037.839879, 361875.322621, 362228.367857, 362456.252380, 362386.932998, 362223.660085, 361947.599188, 362267.821995, 362321.213830, 361994.792704, 362028.156096, 362281.808594, 362118.512883, 362215.700919, 361936.085613, 361965.515258, 361953.341204, 361577.849000, 361232.960507, 361352.722846, 361336.304323, 361145.479964, 361422.596818, 361487.723590, 361186.616871, 360909.531099, 360761.398948, 360855.496790, 361397.483112, 361363.069475, 361075.068046, 361014.256778, 360829.101892, 360257.343393, 359921.960283, 359108.783606, 358435.418272, 358370.041783, 358645.953018, 358267.695310, 357740.574531, 357120.912039, 357565.515641, 359492.868920, 362711.631335, 365419.189629, 362888.321160, 354791.416247, 349733.909105, 353338.093700, 355916.888546, 355929.887298, 355582.289302, 355630.483197, 355663.976845, 355584.108196, 356127.016718, 355970.261925, 355882.091592, 356132.177183, 356060.295110, 356209.595709, 356671.349528, 356915.919128, 356975.003353, 357536.237421, 358222.583616, 358998.700961, 358783.310577, 357786.168035, 357487.366863, 356156.695363, 354391.518794, 352758.356554, 351562.759450, 350761.731273, 350035.992417, 349158.187972, 348697.354677, 348523.968711, 348284.214461, 348009.481488, 347954.268082, 347891.976564, 347619.214164, 347456.754797, 346831.700709, 346616.579410, 346833.256970, 346567.434689, 346258.290571, 346558.791463, 346649.620575, 346563.997144, 345789.459891, 345661.577316, 346050.959755, 345905.710567, 345349.195462, 344854.939409, 344657.771353, 344682.743556, 344624.121624, 344834.749853, 344703.524467, 344156.719492, 343904.580796, 343906.093384, 344250.213946, 343922.824730, 343219.861482, 343222.000677, 343827.719913, 343550.578098, 342883.775406, 343023.279114, 343469.721069, 343222.662666, 343468.137327, 343862.155430, 344202.096059, 344077.837576, 343876.588706, 343866.208211, 343640.590073, 343110.034298, 342737.559048, 342291.497084, 342301.782195, 342716.284300, 342740.851652, 342712.527901, 342230.446616, 342273.883175, 343687.250740, 346965.994447, 350031.274790, 349705.195636, 343621.596991, 336614.551292, 335559.519006, 339377.738381, 342546.183325, 344043.485445, 344143.094975, 343852.459300, 343871.569144, 344209.379371, 344817.154329, 345245.310703, 345607.913944, 346236.860426, 346723.307702, 347134.364217, 347577.519933, 348079.315632, 348616.255633, 349980.642657, 350802.938348, 350917.831110, 350983.981296, 351088.817481, 350735.239276, 350032.876030, 349022.467288, 347365.156898, 345626.083778, 345178.903165, 345387.298733, 344398.566112, 343948.491696, 344884.282498, 344752.107525, 344260.122267, 344642.229328, 345248.811054, 344442.190303, 344503.231389, 345015.165295, 344621.071615, 344604.765285, 345381.792191, 345634.999165, 345753.201371, 345931.841222, 346043.825412, 346184.497896, 345991.544313, 346008.269021, 346221.808481, 346641.660691, 346624.881934, 346492.324850, 346787.908075, 347221.546223, 347203.897230, 347006.349135, 346860.718513, 347206.519275, 347134.349770, 347475.880402, 347655.477701, 347171.934375, 346901.360048, 346982.978849, 347174.859897, 347411.732960, 347710.307985, 347810.338689, 347773.871277, 347648.546206, 347468.293083, 347956.141039, 348978.712002, 348987.335749, 348988.701946, 349439.633977, 349267.031165, 348285.953877, 348357.613136, 348626.515815, 348195.046619, 348042.443227, 348726.926710, 348966.080837, 348895.689291, 349434.152668, 352020.095376, 355948.780962, 358036.988938, 354579.943900, 346850.710526, 342604.532431, 344917.782699, 348762.414825, 349990.049519, 350732.991982, 351539.164310, 351747.192649, 351790.411670, 352351.703202, 352809.119963, 353053.235318, 353170.312704, 353250.079841, 353790.064508, 355032.797669, 355938.378133, 356711.439319, 357440.641746, 358216.736792, 359279.968579, 359983.803489, 360334.163666, 360425.533214, 359882.944510, 358773.127856, 357689.270086, 356895.771382, 356094.150605, 355246.909265, 354284.303288, 354435.171541, 354357.508410, 354487.487903, 354977.023898, 354871.258767, 355063.798013, 355611.334577, 355097.173679, 354707.860246, 355276.012838, 355671.881776, 356100.644853, 356384.745366, 356161.471606, 356513.446211, 357100.441495, 357383.789708, 357058.903419, 357003.907610, 357247.204137, 358164.215515, 358459.571761, 357847.962705, 357972.422506, 358794.826659, 359157.358135, 358862.853189, 358745.198813, 359326.401737, 359998.029074, 360472.444314, 360406.051013, 360379.457538, 361530.934396, 362110.489719, 362248.195040, 362675.008515, 363121.898228, 363013.779425, 363028.843431, 362836.763006, 362530.037659, 362457.378740, 362543.238602, 362081.563319, 362109.141831, 362122.011598, 361544.991107, 361415.304143, 362495.392966, 365889.239050, 370083.445865, 370691.461270, 365214.780673, 357304.001193, 354169.273208, 357168.652716, 360351.045682, 362045.564453, 362856.031666, 363019.518438, 363054.780290, 363707.540157, 363947.036125, 363944.270743, 363933.800752, 364267.791162, 364561.397155, 365058.173435, 365768.387091, 366461.412959, 366856.303108, 367331.248236, 368380.918470, 369207.754063, 370064.073696, 370141.437520, 369309.725970, 368828.051588, 368052.976095, 366446.711202, 365635.086628, 364456.787804, 363146.745311, 362607.099113, 362503.390037, 362116.993954, 362277.245981, 362429.292575, 362224.720369, 362018.809290, 362522.975087, 362722.538135, 362768.322579, 363607.933330, 363500.741605, 363823.082716, 364760.290288, 365667.055112, 366325.715890, 367110.368970, 368634.495145, 369701.726966, 371116.835925, 372886.742030, 374586.553905, 375927.287576, 377153.081120, 378742.152718, 380692.891264, 382085.855075, 382792.443624, 383074.960549, 383631.256114, 384143.742060, 384026.814905, 383623.749380, 382895.253838, 382373.609850, 382649.012185, 382874.176319, 382566.852299, 382079.950082, 382426.725200, 382990.578405, 382806.972686, 382453.704398, 382509.718878, 382395.810905, 381989.316642, 381388.525226, 380626.789274, 379910.392416, 379678.078501, 379873.195145, 379987.382292, 379970.998945, 379089.202747, 378546.052441, 379305.690308, 382194.151588, 385799.383557, 387451.358069, 383383.624461, 375330.556513, 371449.600909, 374009.189121, 377523.397014, 379208.538678, 379389.109232, 379510.885035, 379721.463341, 380029.201920, 380282.720200, 379993.734084, 380055.707520, 380137.765436, 380010.824403, 380390.150644, 380969.184742, 381465.737172, 382306.565068, 382288.635844, 382466.051730, 382862.569127, 382913.720163, 382169.961950, 381145.011693, 380502.877629, 379423.646593, 377199.774255, 375759.722792, 374207.066052, 372330.829157, 371746.077069, 371600.385254, 371232.105624, 370863.156921, 370777.041252, 370663.186411, 370645.680660, 370877.583862, 370964.938177, 370653.081596, 370710.307702, 370528.046749, 370148.453320, 370224.510050, 370936.244111, 371134.383657, 371207.047898, 370775.768960, 370620.745630, 370756.480044, 370540.641480, 370367.723566, 370444.408606, 370709.563027, 370493.215399, 370068.126298, 370416.345951, 370222.984823, 369615.659539, 369630.775753, 369798.517309, 369607.060955, 369745.276150, 369697.517895, 369507.276475, 369233.364013, 369309.703345, 369531.210794, 369301.694223, 369037.197281, 369279.790984, 369147.715476, 369301.667730, 369824.186275, 369814.460346, 369900.483993, 369949.860119, 369616.884645, 369061.081331, 368288.082015, 368015.326272, 368021.157794, 367813.047134, 367562.548002, 367879.564157, 367923.131069, 367912.996670, 369338.721423, 372609.417294, 375675.979479, 374471.141145, 367508.061826, 360643.110623, 360293.704707, 364581.983639, 367557.893334, 368034.463911, 368443.901415, 368623.186721, 369031.108681, 369052.798678, 369175.876137, 369460.444665, 369627.136271, 370009.181951, 370371.401771, 370591.447274, 371463.626421, 372241.703381, 372874.151706, 373645.139903, 374436.458792, 374918.130655, 375370.764635, 375645.316915, 375195.553646, 374510.199105, 373554.058653, 372873.849274, 371312.861041, 369572.589937, 369057.971806, 369161.959442, 368508.321964, 368206.892479, 368213.924239, 368427.642436, 368191.579606, 367808.244129, 368219.177872, 368395.537327, 367917.920334, 367891.381590, 368057.682925, 367904.181401, 367940.773240, 368430.407442, 368577.840321, 367906.571349, 367937.352136, 368410.936430, 368553.651016, 368218.613614, 367862.740669, 368099.635077, 368373.235788, 368035.133793, 367975.104483, 367806.262780, 368259.338223, 368267.878439, 367914.088527, 368213.786624, 368253.258782, 367699.056471, 367670.528283, 367893.881235, 368070.279484, 368619.046044, 368949.730325, 368619.520123, 368245.218904, 368571.145363, 368628.301782, 368557.020580, 368583.576292, 368903.572026, 368986.685268, 369375.197938, 369423.155699, 369398.427813, 369571.258824, 369306.028978, 368478.858305, 368055.706278, 368297.612346, 368191.959828, 367673.086474, 367127.515912, 367024.466553, 367698.563469, 369302.887198, 372637.132295, 375762.741630, 375045.942210, 367873.797216, 360890.017682, 360274.272536, 364247.605370, 367567.443490, 368371.636265, 368302.703639, 369005.414114, 369461.342547, 369422.792587, 369541.386820, 370013.981378, 370440.379898, 370427.564699, 370507.059543, 370920.189582, 371850.231867, 372413.344985, 372593.034859, 373371.250160, 374836.602138, 375317.922942, 375784.950542, 375898.564068, 375518.563573, 374668.151272, 373527.167060, 372213.432321, 370774.428676, 369707.976724, 369154.960444, 368391.146378, 367770.872062, 367262.732191, 367236.857832, 367466.941435, 367196.849801, 366875.569970, 366710.070828, 367107.890649, 367138.801717, 367239.689901, 367356.063543, 366757.804484, 366458.267240, 367259.307619, 367346.682779, 367012.609781, 367082.129941, 367386.449786, 366875.078436, 366819.210215, 367113.698818, 366838.803392, 366597.436276, 366663.186555, 366776.250025, 366543.174468, 366459.822514, 366812.620070, 366623.452913, 366061.408671, 366183.424142, 366027.612255, 366108.776287, 366286.702919, 366264.517004, 365987.680762, 365982.191626, 366011.789914, 365912.879615, 365516.717857, 365772.487785, 365958.140776, 365659.091634, 365844.028874, 366461.370010, 366571.677526, 366649.364647, 366851.763194, 366460.321833, 365807.605778, 365483.254675, 365323.603930, 364629.724167, 364176.411407, 364494.668299, 364788.411042, 364439.740500, 364758.273796, 365370.003660, 367469.055519, 371224.264415, 372499.978698, 361564.029625, 356129.716737, 358309.157373, 362134.289384, 364002.616193, 364486.980654, 365153.799554, 365623.243488, 365414.036922, 365665.868683, 365991.904271, 366240.778349, 366258.226786, 366075.316508, 366075.421807, 366814.530385, 367536.395944, 368356.493955, 368805.774548, 369426.837018, 370639.797282, 370806.705331, 371028.457532, 371342.178623, 370437.204642, 368427.027558, 366473.593625, 365112.399871, 364028.226606, 362886.272999, 362220.105463, 362578.074608, 362466.229576, 362198.083771, 361884.835780, 361685.051293, 361287.422043, 361415.874166, 361510.574782, 361365.318285, 361477.842710, 361736.214542, 361604.819315, 361379.292375, 361737.552028, 362086.209094, 361783.331077, 361303.312502, 361536.788970, 361616.374811, 361833.953140, 361696.434786, 361277.512776, 361066.420229, 361466.320241, 361502.484236, 361514.871780, 361090.609545, 360609.268803, 360300.285861, 360721.863667, 360821.210670, 360699.976518, 360833.819001, 360633.415877, 360799.107655, 360766.836052, 360316.717816, 360116.190442, 360176.958920, 360492.910971, 361155.808997, 361407.622031, 361473.893992, 361477.466967, 361857.688002, 360789.906291, 359183.106977, 359280.172221, 359797.851578, 359603.412411, 359506.624068, 359171.853678, 359271.931736, 359258.172737, 359341.767543, 361757.120573, 365745.179278, 367842.377759, 364578.556403, 356184.974513, 350645.637204, 352760.953363, 356902.513185, 359047.838228, 359420.802402, 359646.427775, 360402.771130, 361018.584092, 360513.249468, 360535.127182, 360998.811497, 360931.537653, 361757.311793, 362392.909667, 362517.993334, 363153.854398, 363924.862647, 364541.371295, 365063.925236, 365916.922051, 366499.909708, 366541.210445, 366250.527833, 365038.967833, 363360.479871, 362300.918262, 360880.352564, 359959.210188, 358733.175548, 357919.901898, 358079.620377, 358163.802966, 357596.185024, 357012.220161, 357071.478090, 357108.588239, 357003.833131, 357059.392416, 357055.189756, 357319.250217, 357177.396929, 357063.064457, 357459.791684, 357395.720851, 356927.088459, 357511.734002, 357736.718068, 357207.733816, 356555.754223, 356723.179835, 357003.088134, 357187.790610, 356957.595364, 356766.160432, 357072.869526, 356881.100106, 356668.644006, 356697.001240, 356593.135210, 356238.234590, 355973.119764, 356227.811089, 357047.453260, 357384.420207, 357486.356761, 357110.405870, 357252.600540, 357196.295426, 356283.474925, 355188.975981, 354952.070827, 355183.520687, 355541.938337, 355502.417981, 355314.136006, 354890.607202, 354611.196728, 356127.140712, 359514.019886, 363000.777590, 363110.251107, 356916.229293, 348940.583991, 347218.712799, 350906.869217, 354259.158407, 355651.067200, 356114.897407, 355592.477500, 355891.227389, 356434.846463, 356450.844936, 356668.531822, 357274.006863, 357221.365716, 357230.235045, 357975.945965, 358748.077855, 358996.117836, 359479.376723, 360186.055633, 361092.449274, 361972.976451, 362344.632941, 362549.615415, 361767.588501, 360917.285751, 360391.700533, 359014.120105, 357147.419076, 355708.208353, 354578.732538, 354467.370984, 353961.965163, 353075.633153, 353172.638520, 353001.741349, 352816.855235, 352672.320780, 352489.298001, 352102.484216, 352197.156383, 352691.366357, 352283.924138, 351957.235171, 351921.505562, 352092.999280, 352075.181636, 351817.470291, 351612.101136, 351617.575392, 351310.499650, 351205.173074, 350914.255596, 350660.247320, 350434.827510, 350061.371741, 349887.480375, 350016.014011, 349988.347957, 349633.253533, 349394.182718, 349681.415510, 349365.862631, 348774.712434, 348776.986440, 349226.191663, 349144.080788, 349498.415020, 350016.042722, 350470.999204, 350729.354378, 350712.225508, 350298.938916, 349759.971629, 349309.496415, 349277.629278, 349229.121426, 349401.790831, 349764.442408, 349652.664031, 349689.500464, 350136.094372, 351243.994972, 353778.638986, 357286.335204, 359589.768483, 357299.848011, 350403.561609, 346005.476573, 347724.655692, 352392.389615, 354923.764075, 355813.700634, 356372.822717, 356865.390418, 357271.508834\n};\n\n#endif /* SAMPLE_SIGNALS_H */\n')
print(f"Generated sample signals header: {os.path.join(deploy_dir, 'sample_signals.h')}")

# 2. Write deploy/predict_bp_example.c
with open(os.path.join(deploy_dir, "predict_bp_example.c"), "w") as f:
    f.write('/**\n * @file predict_bp_example.c\n * @brief Complete ANSI C Demonstration Program for Cuff-Less Blood Pressure Prediction (SBP & DBP)\n */\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <stdbool.h>\n#include <math.h>\n#include <string.h>\n\n#include "bp_pipeline.h"\n#include "sample_signals.h"\n\nstatic void print_banner(const char *title) {\n    printf("\\n================================================================================\\n");\n    printf("   %s\\n", title);\n    printf("================================================================================\\n");\n}\n\nint main(int argc, char *argv[]) {\n    (void)argc; (void)argv;\n\n    print_banner("CUFF-LESS BLOOD PRESSURE INFERENCE ENGINE (ANSI C / m2cgen)");\n    printf("Firmware Target   : Standalone ANSI C Embedded Inference (SBP & DBP)\\n");\n    printf("Model Architecture: LightGBM Regressor Decision Trees (Pure C, Zero Dependencies)\\n");\n    printf("Feature Vector    : %d Physiological Biomarkers (Pure Waveforms, NO Age)\\n", NUM_INPUT_FEATURES);\n\n    const double *red_data = SAMPLE_PPG_RED;\n    const double *ir_data  = SAMPLE_PPG_IR;\n    const double *ecg_data = SAMPLE_ECG_LEAD_I;\n    size_t data_length     = SAMPLE_SIGNAL_LEN;\n    double is_male         = SAMPLE_IS_MALE;\n\n    printf("\\nTesting on Subject %s from Clinical Dataset (10.0s Window @ 100 Hz)...\\n", SAMPLE_SUBJECT_ID);\n    printf("[1/4] Running 100 Hz Biquad Filters (PPG 0.2-10Hz, ECG 0.5-35Hz)...\\n");\n    printf("[2/4] Detecting QRS complexes, systolic peaks, feet, and maximum velocities...\\n");\n    printf("[3/4] Extracting %d multi-domain physiological features...\\n", NUM_INPUT_FEATURES);\n    printf("[4/4] Executing LightGBM SBP and DBP Decision Tree Ensembles in C...\\n");\n\n    bp_prediction_result_t bp_result;\n    bool success = bp_predict_from_raw(red_data, ir_data, ecg_data, data_length, is_male, &bp_result);\n\n    if (!success) {\n        printf("\\n>>> ERROR: Blood pressure estimation failed due to insufficient signal quality. <<<\\n");\n        return 1;\n    }\n\n    print_banner("EXTRACTED PHYSIOLOGICAL BIOMARKERS & TIMING PARAMETERS");\n    printf("  Estimated Heart Rate       : %6.1f BPM (RR interval: %.1f ms)\\n", \n           bp_result.heart_rate_bpm, (60.0 / bp_result.heart_rate_bpm) * 1000.0);\n    printf("  Detected Cardiac Cycles    : %6d beats\\n", bp_result.num_detected_beats);\n    printf("  PAT (R-Peak to Pulse Foot) : %6.2f ms\\n", bp_result.pat_foot_ms);\n    printf("  PAT (R-Peak to Sys Peak)   : %6.2f ms\\n", bp_result.pat_peak_ms);\n    printf("  PTT (Inter-channel Red-IR) : %6.2f ms\\n", bp_result.ptt_inter_peak_ms);\n    printf("  Pulse Width at 50%% Height  : %6.2f ms\\n", bp_result.pulse_width_50_ms);\n    printf("  Vascular Stiffness Ratio   : %6.4f (Tsys / Tdia)\\n", bp_result.stiffness_k_val);\n    printf("  Red/IR Optical Ratio (R)   : %6.4f\\n", bp_result.optical_ratio_r);\n\n    print_banner("ESTIMATED BLOOD PRESSURE INFERENCE RESULTS");\n    printf("  Systolic Blood Pressure  (SBP) : %7.2f mmHg\\n", bp_result.sbp);\n    printf("  Diastolic Blood Pressure (DBP) : %7.2f mmHg\\n", bp_result.dbp);\n    printf("  Mean Arterial Pressure   (MAP) : %7.2f mmHg (Analytically calculated)\\n", bp_result.map_calc);\n    printf("  Pulse Pressure           (PP)  : %7.2f mmHg\\n", bp_result.sbp - bp_result.dbp);\n    printf("  Clinical Category (AHA/ACC)    : %s\\n", bp_result.aha_category);\n\n    printf("\\n--- Clinical Ground Truth Comparison (Subject %s) ---\\n", SAMPLE_SUBJECT_ID);\n    printf("  Reference SBP : %7.2f mmHg (Error: %+.2f mmHg)\\n", SAMPLE_TRUE_SBP, bp_result.sbp - SAMPLE_TRUE_SBP);\n    printf("  Reference DBP : %7.2f mmHg (Error: %+.2f mmHg)\\n", SAMPLE_TRUE_DBP, bp_result.dbp - SAMPLE_TRUE_DBP);\n    printf("  Reference MAP : %7.2f mmHg (Error: %+.2f mmHg)\\n", SAMPLE_TRUE_MAP, bp_result.map_calc - SAMPLE_TRUE_MAP);\n    printf("  Reference HR  : %7.1f BPM  (Diff : %+.1f BPM)\\n", SAMPLE_TRUE_HR, bp_result.heart_rate_bpm - SAMPLE_TRUE_HR);\n\n    print_banner("C FIRMWARE PIPELINE EXECUTION COMPLETED SUCCESSFULLY");\n    return 0;\n}\n')
print(f"Generated standalone example: {os.path.join(deploy_dir, 'predict_bp_example.c')}")

# 3. Write deploy/Makefile
with open(os.path.join(deploy_dir, "Makefile"), "w") as f:
    f.write('CC ?= gcc\nCFLAGS ?= -O3 -Wall -Wextra -std=c99\nLDFLAGS ?= -lm\n\nSRCS = predict_bp_example.c bp_pipeline.c ppg_filter.c ecg_filter.c lgbm_sbp.c lgbm_dbp.c\nTARGET = predict_bp_example\n\nall: predict_bp_example test_deploy\n\npredict_bp_example: $(SRCS)\n\t$(CC) $(CFLAGS) -I. $(SRCS) $(LDFLAGS) -o predict_bp_example\n\ntest_deploy: test_deploy.c lgbm_sbp.c lgbm_dbp.c ppg_filter.c ecg_filter.c\n\t$(CC) $(CFLAGS) -I. test_deploy.c lgbm_sbp.c lgbm_dbp.c ppg_filter.c ecg_filter.c $(LDFLAGS) -o test_deploy\n\ntest: all\n\t./test_deploy\n\t./predict_bp_example\n\nclean:\n\trm -f predict_bp_example test_deploy\n\n.PHONY: all test clean\n')
print(f"Generated Makefile: {os.path.join(deploy_dir, 'Makefile')}")


## Step 4: Compile with GCC & Test Ported C Code on a Clinical Subject


In [ ]:
# 1. Compile and run test suite with make in deploy/
res_make = subprocess.run(f"make -C '{deploy_dir}' clean && make -C '{deploy_dir}' test", shell=True, capture_output=True, text=True)
print(res_make.stdout)
if res_make.returncode != 0:
    print("Compilation/Test Error Output:\n", res_make.stderr)
    raise RuntimeError(f"Makefile compilation/testing failed with exit code {res_make.returncode}")
